# Female population of India and its states by age, 1950–2100

This notebook builds **annual female population by state/UT × 16 five-year age bands (00–04 … 70–74, 75+) × year, 1950–2100**,
for India and all 37 state/UT series used by ICMR-NCDIR. The output is general-purpose: population denominators, age-structured
demographic or epidemiological models, planning, or any analysis that needs consistent age-specific populations over a long horizon.

### Why a new series is needed

No single public source gives this directly:

| Source | Coverage | Limitation |
|---|---|---|
| **ICMR-NCDIR state population projections** | 37 states/UTs, 2012–2036, 16 bands | Only 25 years; each state's **age split is fixed** over the whole period |
| **UN World Population Prospects (WPP) 2024** | India, single ages, 1950–2100 | India only; levels differ from the Census |
| **Census of India 1991, 2001, 2011** | All states, 16 bands | Only three years; boundaries changed since |
| **SRS Statistical Report 2022** | India + 22 bigger states, age distribution | One year; used here only for validation |

### Three constructions ("tracks")

| Track | Totals | Age split | In short |
|---|---|---|---|
| **A** | ICMR-NCDIR, extended with WPP | ICMR-NCDIR's fixed split | Extends the ICMR-NCDIR projections as they are |
| **B** | Census + WPP | Census + WPP, changing over time | Demographic reference anchored to the Census |
| **C** | ICMR-NCDIR, extended with WPP | Track B's split | ICMR-NCDIR totals with a realistic age structure |

### Contents

| Section | What it does |
|---|---|
| 0 | Setup, configuration and data loading |
| 1 | What the ICMR-NCDIR projections contain (and why their age split is fixed) |
| 2 | Track A: extending ICMR-NCDIR to 1950–2100 (four variants) |
| 3 | Cohort diagnostics: does a series behave like a real population? |
| 4 | Track B: Census-anchored series for India and all states |
| 5 | Track C: ICMR-NCDIR totals × Track B age split |
| 6 | Comparison of the three tracks |

Run the sections in order; later sections reuse objects built earlier. Figures go to `figures/` and data outputs to `outputs/` at the repository root (configurable in Section 0).

**Requirements:** Python ≥ 3.10 with `numpy`, `pandas`, `scipy`, `matplotlib`, `openpyxl` and `pyarrow`
(see `requirements.txt`). Helper code lives in `src/popproj.py`; figures use a built-in style (`src/plotstyle.py`) unless the
optional `sciplotstyle` package is installed.

## 0. Setup

Input files are read from `data/raw/` by default; to use other copies, set them here (or through the environment variables `POPPROJ_ICMR_NCDIR`, `POPPROJ_WPP`,
`POPPROJ_CENSUS`, `POPPROJ_SRS2022`). The SRS file is optional and only needed for the validation in Section 4.5.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))   # use the repository's src/ when run from notebooks/
import popproj as px
from popproj import sp, BANDS, YEARS, ICMR_START, ICMR_END

# --- input files and output folders --------------------------------------------
# Defaults point inside this repository: data/raw/ (inputs), figures/ and outputs/.
# To use other copies pass any of: icmr_ncdir=, wpp=, census=, srs2022=, fig_dir=, out_dir=
px.configure()
px.check_paths()

px.setup_style(scale=0.75)
pd.set_option("display.width", 200, "display.max_columns", 40)

### 0.1 Load the data

- **ICMR-NCDIR**: 37 state/UT series, 2012–2036, 16 bands. India = sum of the 37 series.
- **WPP 2024**: India female population by single age, 1950–2100, aggregated to the 16 bands.
- **Census**: India female 1991 (J&K was not enumerated in 1991), 2001, 2011, with age-not-stated redistributed pro rata.

In [ ]:
icmr   = px.load_icmr_ncdir()                                  # (state, year) x band
STATES = icmr.index.get_level_values("state").unique().tolist()
icmr_india = icmr.groupby(level="year").sum()            # year x band
wpp    = px.load_wpp_bands()                             # year x band, 1950-2100
census = px.load_census_india()                          # anchor year x band

print(f"ICMR-NCDIR: {len(STATES)} series, years {icmr_india.index.min()}-{icmr_india.index.max()}")
print(f"India totals (M): ICMR-NCDIR 2012 = {icmr_india.loc[2012].sum()/1e6:.1f} | "
      f"WPP 2012 = {wpp.loc[2012].sum()/1e6:.1f} | Census 2011 = {census.loc[2011].sum()/1e6:.1f}")

---
## 1. What the ICMR-NCDIR projections contain

The ICMR-NCDIR file gives, for each state/UT and year, the female population in 16 bands. Written as a total times age shares,

$$P_{s,b}(t) = N_s(t)\,c_{s,b},$$

the **state totals $N_s(t)$ follow a genuine projection** (each state has its own growth path), but the **age shares $c_{s,b}$ do not
change with time**: every band grows at exactly the same rate as the state total. The cell below checks this and identifies where
each state's fixed split comes from.

In [ ]:
sh_ic = icmr.div(icmr.sum(axis=1), axis=0)
drift = sh_ic.groupby(level="state").agg(lambda x: x.max() - x.min()).max(axis=1) * 100
print(f"Largest change in any band's share, 2012-2036, over all 37 series: {drift.max():.4f} percentage points")

first_split = sh_ic.xs(ICMR_START, level="year")
cen11 = px.load_census_state_shares(STATES, 2011)

def rounded_table(s):
    # is this split a table of percentages printed to one decimal (renormalised)?
    K = min(np.arange(95, 105.0001, 0.1), key=lambda K: np.abs(s * K * 10 - np.round(s * K * 10)).max())
    return np.abs(s * K * 10 - np.round(s * K * 10)).max() < 0.02

rows = []
for s in STATES:
    same = [o for o in STATES if o != s and np.allclose(first_split.loc[o], first_split.loc[s], atol=1e-9)]
    if np.allclose(first_split.loc[s], cen11.loc[s], atol=1e-4):
        src = "own Census 2011 age split"
    elif rounded_table(first_split.loc[s]):
        src = "0.1 %-rounded percentage table" + (f" (shared with {', '.join(same)})" if same else "")
    else:
        src = "other"
    rows.append({"state": s, "source of fixed split": src,
                 "share 0-14 [%]": first_split.loc[s, BANDS[:3]].sum() * 100,
                 "Census 2011 share 0-14 [%]": cen11.loc[s, BANDS[:3]].sum() * 100})
split_source = pd.DataFrame(rows).set_index("state").round(1)
print(split_source["source of fixed split"].str.split(" \\(").str[0].value_counts().to_string())
split_source

**Reading the table.** 30 series use a percentage age table printed to 0.1 % (22 distinct tables — one per larger state; smaller units
such as Puducherry, Lakshadweep, Goa or Ladakh reuse a neighbouring state's table), and 7 smaller states/UTs use their own
Census 2011 split. The fixed tables describe an older population than India in 2012, so the young bands are too small and the
old bands too large early in the period.

### 1.1 Age shares over time: WPP moves, ICMR-NCDIR is fixed

WPP's shares change every year as the population ages; ICMR-NCDIR's are flat for 25 years. Census values are shown for reference.
Where a flat line crosses the WPP curve is roughly the year the fixed split resembles.

In [ ]:
wpp_share   = wpp.div(wpp.sum(axis=1), axis=0) * 100
icmr_share  = icmr_india.div(icmr_india.sum(axis=1), axis=0) * 100
cen_share   = census.div(census.sum(axis=1), axis=0) * 100

fig, axs = plt.subplots(4, 4, figsize=(15, 11))
for ax, b in zip(axs.flat, BANDS):
    px.shade_icmr_window(ax, label=(b == BANDS[0]))
    ax.plot(wpp_share.index, wpp_share[b], color=sp.GREY, lw=1.4, label="WPP India")
    ax.plot(icmr_share.index, icmr_share[b], color=sp.OUTC, lw=2.2, label="ICMR-NCDIR (sum of states)")
    ax.scatter(cen_share.index, cen_share[b], color=sp.L1, s=28, zorder=5, label="Census")
    px.style_pop_axis(ax, title=f"Age band {b}", ylabel="Share of females [%]")
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("1.1  India age shares: WPP drifts, ICMR-NCDIR is frozen", y=1.0)
plt.tight_layout(); px.save(fig, "exp1_1_shares_raw"); plt.show()

---
## 2. Track A — extending ICMR-NCDIR to 1950–2100

ICMR-NCDIR covers 2012–2036 only. Track A extends it backwards to 1950 and forwards to 2100 using WPP India's growth,
keeping ICMR-NCDIR's numbers unchanged inside 2012–2036.

**Ratio method.** For a series $y(t)$ known on 2012–2036 and a WPP reference $W(t)$:

$$y(t) = y(2036)\,\frac{W(t)}{W(2036)}\ \ (t>2036), \qquad y(t) = y(2012)\,\frac{W(t)}{W(2012)}\ \ (t<2012).$$

Levels match at the joins but the **growth rate can jump** there (a kink). The blended version works on log-growth
$g(t)=\ln\frac{y(t)}{y(t-1)}$ and fades from ICMR-NCDIR's growth to WPP's over `T_BLEND` years:

$$g(t) = w(k)\,g_{\text{ICMR-NCDIR, at join}} + \bigl(1-w(k)\bigr)\,g_W(t), \qquad
w(k)=\tfrac12\bigl(1+\cos(\pi k/T)\bigr)\ \text{for}\ k<T,\ \text{else } 0,$$

with $k$ = years from the join.

| Variant | Outside 2012–2036 | Age shares |
|---|---|---|
| **A1** | extend the **total** with WPP India's total growth; split by the fixed shares | fixed everywhere |
| **A2** | extend **each band** with WPP India's band growth | fixed inside, changing outside |
| **A1-blend**, **A2-blend** | as above, with the growth rate blended over `T_BLEND` years | as above |

States have no WPP series, so every state uses India's WPP shape. India = sum of the states.

### 2.1 The extension function

`extend(inside, ref, T)` applies the rule to every column of `inside` (years 2012–2036) using the matching column of `ref` (WPP, 1950–2100).

In [ ]:
def taper(k, T):
    # weight on the ICMR-NCDIR growth rate, k years away from the join
    if T <= 0 or k >= T:
        return 0.0
    return 0.5 * (1 + np.cos(np.pi * k / T))

def extend(inside, ref, T=0, years=YEARS):
    # inside: DataFrame (years 2012-2036) x columns ; ref: DataFrame (1950-2100) with the same columns
    t0, t1 = inside.index.min(), inside.index.max()
    out = pd.DataFrame(index=years, columns=inside.columns, dtype=float)
    out.loc[t0:t1] = inside.values
    g_start = np.log(inside.loc[t0 + 1] / inside.loc[t0])      # ICMR-NCDIR growth at the 2012 join
    g_end   = np.log(inside.loc[t1] / inside.loc[t1 - 1])      # ICMR-NCDIR growth at the 2036 join
    for t in range(t1 + 1, years.max() + 1):                   # forward
        w = taper(t - t1, T)
        gW = np.log(ref.loc[t] / ref.loc[t - 1])
        out.loc[t] = out.loc[t - 1] * np.exp(w * g_end + (1 - w) * gW)
    for t in range(t0 - 1, years.min() - 1, -1):               # backward
        w = taper(t0 - t, T)
        gW = np.log(ref.loc[t + 1] / ref.loc[t])
        out.loc[t] = out.loc[t + 1] / np.exp(w * g_start + (1 - w) * gW)
    return out

T_BLEND = 10    # years over which the growth rate fades from ICMR-NCDIR's to WPP's

### 2.2 Build the four variants for every state; India = sum of states

In [ ]:
wpp_total = wpp.sum(axis=1).to_frame("total")

def build_A1(state, T):
    P = icmr.loc[state]
    N = extend(P.sum(axis=1).to_frame("total"), wpp_total, T)["total"]
    shares = (P.iloc[0] / P.iloc[0].sum())                     # frozen c_{s,b}
    return pd.DataFrame(np.outer(N, shares), index=YEARS, columns=BANDS)

def build_A2(state, T):
    return extend(icmr.loc[state], wpp, T)

VARIANTS = {
    "A1":       lambda s: build_A1(s, 0),
    "A1-blend": lambda s: build_A1(s, T_BLEND),
    "A2":       lambda s: build_A2(s, 0),
    "A2-blend": lambda s: build_A2(s, T_BLEND),
}
VSTYLE = {"A1": dict(color=sp.L1, lw=1.8), "A1-blend": dict(color=sp.L2, lw=1.4, ls="--"),
          "A2": dict(color=sp.OUTC, lw=1.8), "A2-blend": dict(color=sp.ACC, lw=1.4, ls="--")}

state_var = {v: {s: f(s) for s in STATES} for v, f in VARIANTS.items()}            # variant -> state -> DataFrame
india_var = {v: sum(state_var[v][s] for s in STATES) for v in VARIANTS}             # variant -> DataFrame

for v, P in india_var.items():
    print(f"{v:9s} India total: 1950 = {P.loc[1950].sum()/1e6:6.1f}M | 2012 = {P.loc[2012].sum()/1e6:6.1f}M | "
          f"2100 = {P.loc[2100].sum()/1e6:6.1f}M")

### 2.3 India total, 1950–2100

Top: level. Bottom: annual growth — a kink is a jump in growth at 2012 or 2036. Right: zoom on the two joins.

In [ ]:
fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(2, 3, width_ratios=[2.2, 1, 1])
axL, axG = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0])
axZ = [fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[1, 2])]

wtot = wpp.sum(axis=1)
for ax in [axL, axG] + axZ:
    px.shade_icmr_window(ax, label=(ax is axL))
axL.plot(wtot.index, wtot, color=sp.GREY, lw=1.2, ls=":", label="WPP India")
axL.scatter(census.index, census.sum(axis=1), color="black", s=30, zorder=6, label="Census")
for v, P in india_var.items():
    tot = P.sum(axis=1); g = px.growth_pct(tot)
    axL.plot(tot.index, tot, label=v, **VSTYLE[v])
    axG.plot(g.index, g, label=v, **VSTYLE[v])
    for ax, (lo, hi), series in [(axZ[0], (2002, 2022), tot), (axZ[1], (2026, 2046), tot),
                                 (axZ[2], (2002, 2022), g),   (axZ[3], (2026, 2046), g)]:
        m = (series.index >= lo) & (series.index <= hi)
        ax.plot(series.index[m], series[m], marker="o", ms=2.5, **VSTYLE[v])
        ax.set_xlim(lo, hi)
axG.plot(wtot.index, px.growth_pct(wtot), color=sp.GREY, lw=1.2, ls=":")
px.style_pop_axis(axL, "India total female population", ymax=wtot.max())
px.style_pop_axis(axG, "Annual growth of the total", ylabel="Growth [%/yr]")
for ax, t in zip(axZ, ["Zoom: 2012 join (level)", "Zoom: 2036 join (level)",
                       "Zoom: 2012 join (growth)", "Zoom: 2036 join (growth)"]):
    px.style_pop_axis(ax, t, ylabel="Growth [%/yr]" if "growth" in t else "Female pop. [millions]",
                      ymax=None if "growth" in t else wtot.max())
axL.legend(loc="upper left", frameon=False, ncol=2)
plt.tight_layout(); px.save(fig, "exp1_4_india_total"); plt.show()

### 2.4 India, all 16 bands

Four variants per band, Census as black dots, WPP as a dotted reference.

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(15, 11))
for ax, b in zip(axs.flat, BANDS):
    px.shade_icmr_window(ax, label=(b == BANDS[0]))
    ax.plot(wpp.index, wpp[b], color=sp.GREY, lw=1.0, ls=":", label="WPP India")
    for v, P in india_var.items():
        ax.plot(P.index, P[b], label=v, **VSTYLE[v])
    ax.scatter(census.index, census[b], color="black", s=22, zorder=6, label="Census")
    px.style_pop_axis(ax, f"Age band {b}", ymax=max(wpp[b].max(), india_var['A2'][b].max()))
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=7, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("1.5  India female population by band, four extension variants", y=1.0)
plt.tight_layout(); px.save(fig, "exp1_5_india_bands"); plt.show()

### 2.5 Age shares over time

Each line is one band's share of the total (dark = young, light = old). A1 is flat; A2 is flat inside 2012–2036 and bends outside.

In [ ]:
cmap = plt.get_cmap(sp.CAT)
colors = [cmap(i / (len(BANDS) - 1) * 0.9) for i in range(len(BANDS))]
fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for ax, (v, P) in zip(axs.flat, india_var.items()):
    sh = P.div(P.sum(axis=1), axis=0) * 100
    px.shade_icmr_window(ax, label=False)
    for b, c in zip(BANDS, colors):
        ax.plot(sh.index, sh[b], color=c, lw=1.4, label=b)
    ax.plot(wpp_share.index, wpp_share["00-04"], color=sp.GREY, lw=1, ls=":", label="WPP 00-04")
    px.style_pop_axis(ax, f"{v}: India age shares", ylabel="Share of females [%]")
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="center right", bbox_to_anchor=(1.07, 0.5), frameon=False)
plt.tight_layout(); px.save(fig, "exp1_6_india_shares"); plt.show()

### 2.6 Growth by band and year

Annual growth (%/yr). Dashed lines mark 2012 and 2036; a kink is a colour break along a dashed line.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 9), sharex=True, sharey=True)
norm = TwoSlopeNorm(vmin=-4, vcenter=0, vmax=6)
for ax, (v, P) in zip(axs.flat, india_var.items()):
    G = px.growth_pct(P).loc[1951:]
    im = ax.imshow(G.T.values, aspect="auto", cmap="RdBu_r", norm=norm,
                   extent=[G.index.min() - 0.5, G.index.max() + 0.5, len(BANDS) - 0.5, -0.5])
    for t in (ICMR_START + 0.5, ICMR_END + 0.5):
        ax.axvline(t, color="black", lw=0.8, ls="--")
    ax.set_yticks(range(len(BANDS))); ax.set_yticklabels(BANDS)
    ax.set_title(f"{v}: annual growth by band"); ax.set_ylabel("Age band")
    if ax in axs[1]: ax.set_xlabel("Year")
    px.sp.lock_ticks(ax, "y")
fig.colorbar(im, ax=axs, shrink=0.8, label="Growth [%/yr]")
px.save(fig, "exp1_7_growth_heatmaps"); plt.show()

### 2.7 Kink size at each join

Growth just after the join minus growth just before it (percentage points): $g(2012\to2013)-g(2011\to2012)$ and $g(2036\to2037)-g(2035\to2036)$. Zero = smooth.

In [ ]:
rows = {}
for v, P in india_var.items():
    g = px.growth_pct(P)
    tot = px.growth_pct(P.sum(axis=1))
    rows[(v, "2012 join")] = pd.concat([g.loc[2013] - g.loc[2012], pd.Series({"TOTAL": tot[2013] - tot[2012]})])
    rows[(v, "2036 join")] = pd.concat([g.loc[2037] - g.loc[2036], pd.Series({"TOTAL": tot[2037] - tot[2036]})])
kinks = pd.DataFrame(rows).T.round(2)
kinks["max |kink|"] = kinks[BANDS].abs().max(axis=1)
kinks

### 2.8 Age pyramids at selected years (India)

Shaded bars = A1 (same shape every year); lines = other variants; dotted = WPP; dots = Census.

In [ ]:
PYR_YEARS = [1950, 1991, 2011, 2036, 2070, 2100]
fig, axs = plt.subplots(2, 3, figsize=(15, 9), sharey=True)
y = np.arange(len(BANDS))
for ax, yr in zip(axs.flat, PYR_YEARS):
    ax.barh(y, india_var["A1"].loc[yr] / 1e6, color=sp.L1, alpha=0.25, label="A1")
    for v in ["A1-blend", "A2", "A2-blend"]:
        ax.plot(india_var[v].loc[yr] / 1e6, y, marker="o", ms=3, label=v, **VSTYLE[v])
    ax.plot(wpp.loc[yr] / 1e6, y, color=sp.GREY, lw=1, ls=":", marker="s", ms=2.5, label="WPP India")
    if yr in census.index:
        ax.scatter(census.loc[yr] / 1e6, y, color="black", s=14, zorder=6, label="Census")
    ax.set_yticks(y); ax.set_yticklabels(BANDS)
    ax.set_title(f"{yr}"); ax.set_xlabel("Female pop. [millions]"); ax.set_ylabel("Age band")
    ax.grid(alpha=0.25)
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("1.9  India female age pyramids", y=1.0)
plt.tight_layout(); px.save(fig, "exp1_9_pyramids"); plt.show()

### 2.9 Comparison with the Census (India)

Percent difference from the Census by band (Census 1991 excludes J&K, about 1 % of the total).

In [ ]:
rows = {}
for v, P in india_var.items():
    for yr in census.index:
        d = (P.loc[yr] / census.loc[yr] - 1) * 100
        d["TOTAL"] = (P.loc[yr].sum() / census.loc[yr].sum() - 1) * 100
        rows[(v, yr)] = d
vs_census = pd.DataFrame(rows).T.round(1)
vs_census

### 2.10 States: total population, four variants

In [ ]:
ncol = 6; nrow = int(np.ceil(len(STATES) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(18, 2.6 * nrow))
for ax, s in zip(axs.flat, STATES):
    px.shade_icmr_window(ax, label=False)
    for v in VARIANTS:
        tot = state_var[v][s].sum(axis=1)
        ax.plot(tot.index, tot, label=v, **VSTYLE[v])
    top = max(state_var[v][s].sum(axis=1).max() for v in VARIANTS)
    px.style_pop_axis(ax, s, ylabel="Females", ymax=top)
    ax.title.set_fontsize(9)
for ax in list(axs.flat)[len(STATES):]:
    ax.axis("off")
h, l = axs.flat[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01), frameon=False)
fig.suptitle("1.11  State totals, 1950-2100, four variants", y=1.0)
plt.tight_layout(); px.save(fig, "exp1_11_states_totals"); plt.show()

### 2.11 States: annual growth of the total (state × year)

A1 (left) vs A1-blend (right), states sorted by their 2012–2036 growth.

In [ ]:
order = sorted(STATES, key=lambda s: icmr.loc[s].sum(axis=1).iloc[-1] / icmr.loc[s].sum(axis=1).iloc[0])
fig, axs = plt.subplots(1, 2, figsize=(16, 10), sharey=True)
norm = TwoSlopeNorm(vmin=-2, vcenter=0, vmax=4)
for ax, v in zip(axs, ["A1", "A1-blend"]):
    G = pd.DataFrame({s: px.growth_pct(state_var[v][s].sum(axis=1)) for s in order}).loc[1951:]
    im = ax.imshow(G.T.values, aspect="auto", cmap="RdBu_r", norm=norm,
                   extent=[G.index.min() - 0.5, G.index.max() + 0.5, len(order) - 0.5, -0.5])
    for t in (ICMR_START + 0.5, ICMR_END + 0.5):
        ax.axvline(t, color="black", lw=0.8, ls="--")
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=8)
    ax.set_title(f"{v}: annual growth of state totals"); ax.set_xlabel("Year")
fig.colorbar(im, ax=axs, shrink=0.8, label="Growth [%/yr]")
px.save(fig, "exp1_12_states_growth_heatmap"); plt.show()

### 2.12 States: size of the growth kink at each join

In [ ]:
rows = []
for s in STATES:
    r = {"state": s}
    for v in ["A1", "A1-blend"]:
        g = px.growth_pct(state_var[v][s].sum(axis=1))
        r[f"{v} 2012"] = g[2013] - g[2012]
        r[f"{v} 2036"] = g[2037] - g[2036]
    rows.append(r)
state_kinks = pd.DataFrame(rows).set_index("state").round(2)
state_kinks.reindex(state_kinks["A1 2012"].abs().sort_values(ascending=False).index)

### 2.13 Results

- **A1** has almost no kink for India (growth changes by −0.05 pp at 2012 and −0.06 pp at 2036), but some small units have large
  ones (Dadra & Nagar Haveli +3.6 / −4.8 pp, Daman & Diu +2.7 / −4.2 pp) because their projected growth is far from India's.
- **A2** has band-level kinks up to about 4 pp/yr. In 2.6 the diagonal cohort waves carried by WPP stop at 2012 and restart at 2036:
  nothing ages inside the fixed-split window.
- **Blending** (`T_BLEND = 10`) removes the kinks at the joins for both A1 and A2 (all below 0.1 pp).
- **Totals** stay close to the Census for every variant (A1: +2.2 % in 1991, +0.4 % in 2011), but **age bands do not**
  (A1 1991: 00–04 −39 %, 75+ +152 %), because the fixed split is carried back in time.

### 2.14 Save Track A outputs

Written to `outputs/trackA/`: India series for all four variants (one sheet each) and state series (long parquet) per variant.

In [ ]:
OUTA = os.path.join(px.OUT_DIR, "trackA")
os.makedirs(OUTA, exist_ok=True)
with pd.ExcelWriter(os.path.join(OUTA, "trackA_india.xlsx")) as xw:
    for v, P in india_var.items():
        out = P.copy(); out.index.name = "Year"; out["Total"] = out.sum(axis=1); out.to_excel(xw, sheet_name=v)
for v in VARIANTS:
    long = pd.concat(state_var[v], names=["state", "year"]).reset_index().melt(
        id_vars=["state", "year"], var_name="band", value_name="females")
    long.to_parquet(os.path.join(OUTA, f"trackA_states_{v}.parquet"), index=False)
print("saved to", OUTA); print(sorted(os.listdir(OUTA)))

---
## 3. Cohort diagnostics: does a series behave like a real population?

A **cohort** is a group of women born in the same years — e.g. those aged 00–04 in 1991, who are 05–09 in 1996, 10–14 in 2001, and so on.
Without migration a cohort can only **shrink** as it ages. If a series makes a cohort *grow*, the only way a births → aging → deaths
model can reproduce it is with **negative death rates**. These tests show whether a series is suitable for fitting such models,
not only for use as a set of population counts.

A series with a fixed age split fails this test by construction: the age split never moves, so small young cohorts appear to grow
into larger older bands.

### 3.1 Age structure against the Census: pyramids and ratios

Top: pyramids at the three Census years. Bottom: each series divided by the Census, band by band (1.0 = perfect).

In [ ]:
import os
SERIES_2 = {"Census":         (census,          dict(color="black", marker="o", lw=0, ms=4)),
            "WPP India":      (wpp,             dict(color=sp.GREY, ls=":", marker="s", ms=2.5, lw=1)),
            "A1":             (india_var["A1"], dict(color=sp.L1, marker="o", ms=3, lw=1.6)),
            "A2":             (india_var["A2"], dict(color=sp.OUTC, marker="o", ms=3, lw=1.6))}

fig, axs = plt.subplots(2, 3, figsize=(15, 10))
y = np.arange(len(BANDS))
for j, yr in enumerate([1991, 2001, 2011]):
    ax = axs[0, j]
    for name, (df, st) in SERIES_2.items():
        ax.plot(df.loc[yr] / 1e6, y, label=name, **st)
    ax.set_yticks(y); ax.set_yticklabels(BANDS); ax.set_title(f"{yr}: female age pyramid")
    ax.set_xlabel("Female pop. [millions]"); ax.set_ylabel("Age band"); ax.grid(alpha=0.25)
    ax = axs[1, j]
    w = 0.25
    for i, name in enumerate(["A1", "A2", "WPP India"]):
        df, st = SERIES_2[name]
        ax.barh(y + (i - 1) * w, df.loc[yr] / census.loc[yr], height=w, color=st["color"], label=name)
    ax.axvline(1, color="black", lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(BANDS); ax.set_title(f"{yr}: series ÷ Census")
    ax.set_xlabel("Ratio to Census [-]"); ax.set_ylabel("Age band"); ax.grid(alpha=0.25)
    sp.lock_ticks(ax, "y")
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02), frameon=False)
plt.tight_layout(); px.save(fig, "exp2_1_pyramids_vs_census"); plt.show()

tot = pd.DataFrame({name: (df.loc[[1991, 2001, 2011]].sum(axis=1) / census.sum(axis=1) - 1) * 100
                    for name, (df, _) in SERIES_2.items() if name != "Census"}).round(1)
print("Total female population vs Census [% difference]  (Census 1991 excludes J&K, about 1%)")
tot

### 3.2 Following one cohort through time

Each curve follows one cohort diagonally through the bands, relative to its starting size (100 %). Above 100 % = the cohort grows.
The Census itself shows some growth at young ages because young children are under-counted and ages are misreported; WPP corrects for this.

In [ ]:
def cohort_path(P, start_year, start_band=0, end_year=2100):
    # size of the cohort that is in `start_band` in `start_year`, followed every 5 years (stops before 75+)
    out = {}
    j = 0
    while start_band + j < len(BANDS) - 1 and start_year + 5 * j <= end_year:
        t = start_year + 5 * j
        if t in P.index:
            out[t] = P.loc[t, BANDS[start_band + j]]
        j += 1
    s = pd.Series(out, dtype=float)
    return s / s.iloc[0] * 100 if len(s) else s

COHORTS = [(1991, 0, "Aged 00-04 in 1991 (born ~1987-91)"),
           (1991, 4, "Aged 20-24 in 1991 (born ~1967-71)"),
           (2012, 0, "Aged 00-04 in 2012 (born ~2008-12)")]
fig, axs = plt.subplots(1, 3, figsize=(16, 5.2))
for ax, (t0, b0, title) in zip(axs, COHORTS):
    px.shade_icmr_window(ax, label=(ax is axs[0]))
    for name, (df, st) in SERIES_2.items():
        p = cohort_path(df, t0, b0)
        if len(p) > 1:
            ax.plot(p.index, p.values, label=name, **st)
    ax.axhline(100, color="black", lw=0.8)
    ax.axhspan(100, 200, color=sp.OUTC, alpha=0.05, lw=0)
    ax.text(0.02, 0.96, "above 100 %: cohort grows\n= people from nowhere", transform=ax.transAxes,
            va="top", fontsize=8, color=sp.OUTC)
    ax.set_ylim(40, 180)
    px.style_pop_axis(ax, title, ylabel="Cohort size [% of starting size]")
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.04), frameon=False)
plt.tight_layout(); px.save(fig, "exp2_2_cohort_paths"); plt.show()

### 3.3 Ten-year survival for every age group, and the death rate it implies

For each starting band $b$: $S_b = P_{b+2}(t+10)/P_b(t)$, the fraction of the cohort still present ten years later, and $\mu_b=-\ln(S_b)/10$, the implied average annual death rate. $S_b>1 \Leftrightarrow \mu_b<0$.

In [ ]:
def cohort_survival(P, t):
    # 10-year survival from band b at year t to band b+2 at year t+10, for b = 00-04 ... 60-64
    return pd.Series({BANDS[b]: P.loc[t + 10, BANDS[b + 2]] / P.loc[t, BANDS[b]]
                      for b in range(len(BANDS) - 3)})

SURV = {"Census 1991→2001": (census, 1991, dict(color="black", ls="-", marker="o")),
        "Census 2001→2011": (census, 2001, dict(color="black", ls="--", marker="o")),
        "WPP 1991→2001":    (wpp, 1991,    dict(color=sp.GREY, ls=":", marker="s")),
        "WPP 2012→2022":    (wpp, 2012,    dict(color=sp.GREY, ls="-", marker="s")),
        "A1 1991→2001":     (india_var["A1"], 1991, dict(color=sp.L1, ls="-", marker="o")),
        "ICMR-NCDIR 2012→2022":  (icmr_india, 2012, dict(color=sp.OUTC, ls="-", marker="D")),
        "ICMR-NCDIR 2022→2032":  (icmr_india, 2022, dict(color=sp.OUTC, ls="--", marker="D"))}
surv = pd.DataFrame({k: cohort_survival(df, t) for k, (df, t, _) in SURV.items()})
mu_implied = -np.log(surv) / 10

fig, axs = plt.subplots(1, 2, figsize=(16, 5.5))
x = np.arange(len(surv))
for k, (_, _, st) in SURV.items():
    axs[0].plot(x, surv[k], label=k, ms=4, lw=1.4, **st)
    axs[1].plot(x, mu_implied[k] * 1000, label=k, ms=4, lw=1.4, **st)
axs[0].axhline(1, color="black", lw=0.8); axs[0].axhspan(1, 2, color=sp.OUTC, alpha=0.06, lw=0)
axs[0].set_ylim(0.6, 1.6)
axs[1].axhline(0, color="black", lw=0.8); axs[1].axhspan(-60, 0, color=sp.OUTC, alpha=0.06, lw=0)
axs[1].set_ylim(-40, 40)
for ax, yl, t in [(axs[0], "10-year survival ratio [-]", "Share of a cohort still present 10 years later"),
                  (axs[1], "Implied death rate [per 1000 per yr]", "Implied annual death rate (shaded = negative)")]:
    ax.set_xticks(x); ax.set_xticklabels([f"{b}→+10y" for b in surv.index], rotation=45, ha="right")
    ax.set_xlabel("Starting age band"); ax.set_ylabel(yl); ax.set_title(t); ax.grid(alpha=0.25)
    sp.lock_ticks(ax, "x")
axs[0].legend(loc="upper left", frameon=False, fontsize=8, ncol=2)
plt.tight_layout(); px.save(fig, "exp2_3_cohort_survival"); plt.show()
(mu_implied * 1000).round(1).T

### 3.4 Results

- No series produces negative *populations*; the issue is cohort growth.
- Under A1, women aged 00–04 in 1991 grow to about 178 % of their starting size by 2016. Inside ICMR-NCDIR's own 2012–2036 window,
  the cohort aged 00–04 in 2012 grows to about 148 % by 2036 — a property of the fixed split itself.
- Implied death rates: A1 about −26 per 1000 per year at ages 0–19; ICMR-NCDIR 2012→2022 about −19 per 1000; WPP positive everywhere.
- **Implication:** Track A is usable as a set of population counts, but a births → aging → deaths model fitted to it is forced
  into negative mortality at young ages.

---
## 4. Track B — Census-anchored population, 1950–2100

**Goal:** a demographically consistent female population by state × band × year that passes through the Census and inherits
WPP's year-to-year dynamics. Track B does not try to reproduce ICMR-NCDIR's totals.

| Part | Question |
|---|---|
| 4.1 Census anchors | Clean, boundary-harmonised Census 1991 / 2001 / 2011 for all 37 units |
| 4.2 National series | Band-wise ratio method $P_b(t)=W_b(t)\,\rho_b(t)$: hold vs taper vs band-smoothed $\rho$ |
| 4.3 State age evolution | How do state age structures change 1991→2011? Which extrapolation rule predicts best? |
| 4.4 State series | Each state = national band × the state's share of it, raked so states add up to India |
| 4.5 Validation | Which version best matches an independent 2022 age distribution (SRS)? |
| 4.6 Totals vs ICMR-NCDIR | How far are Track B's totals from ICMR-NCDIR's? |
| 4.7 Outputs | Save the Track B series |

Colours: WPP dotted grey, Census black dots, ICMR-NCDIR amber, Track A indigo, Track B variants turquoise / maroon / amber dashed.

### 4.1 Census anchors harmonised to today's 37 units

Today's boundaries are applied to every Census year:

| Issue | Years | Treatment |
|---|---|---|
| Age not stated | all | Redistributed pro rata across the 16 bands, per state and year |
| J&K not enumerated | 1991 | J&K 2001 × (rest-of-India band growth 1991→2001), band by band |
| Chhattisgarh, Jharkhand, Uttarakhand inside MP, Bihar, UP | 1991 | Parent split band by band using the child's 2001 share of (parent + child) |
| Telangana inside Andhra Pradesh | all | Split by the ICMR-NCDIR 2012 total ratio (same age split for both) |
| Ladakh inside J&K | all | Split by the ICMR-NCDIR 2012 total ratio (same age split for both) |
| Census date 1 March vs mid-year | all | Shift +4 months at each band's intercensal growth (toggle `DATE_SHIFT`) |

India anchor = sum of the 37 harmonised units.

In [ ]:
import os
cen_long = px._census_long()
ANCHOR_YEARS = [1991, 2001, 2011]
DATE_SHIFT = True

# 1) raw 16-band counts per Census unit, age-not-stated redistributed pro rata
RAW = {}
for yr in ANCHOR_YEARS:
    units = [s for s in cen_long[cen_long.Year == yr].state.unique() if not s.startswith("India")]
    RAW[yr] = pd.DataFrame({s: px._census_bands(cen_long, s, yr) for s in units}).T

nd12 = icmr.xs(2012, level="year").sum(axis=1)                # ICMR-NCDIR 2012 state totals (for splits)
SPLIT_ICMR = {"Telangana": "Andhra Pradesh", "Ladakh": "Jammu & Kashmir"}
SPLIT_2001 = {"Chattisgarh": "Madhya Pradesh", "Jharkhand": "Bihar", "Uttarakhand": "Uttar Pradesh"}
adj_log = []

def harmonise(yr):
    D = RAW[yr].copy()
    if yr == 1991:
        rest91 = D.sum()
        rest01 = RAW[2001].drop(index="Jammu & Kashmir").sum()
        D.loc["Jammu & Kashmir"] = RAW[2001].loc["Jammu & Kashmir"] * rest91 / rest01
        adj_log.append((yr, "J&K imputed (M)", D.loc["Jammu & Kashmir"].sum() / 1e6))
        for child, parent in SPLIT_2001.items():
            r = RAW[2001].loc[child] / (RAW[2001].loc[child] + RAW[2001].loc[parent])
            D.loc[child] = D.loc[parent] * r
            D.loc[parent] = D.loc[parent] * (1 - r)
            adj_log.append((yr, f"{child} split from {parent} (share of total)", D.loc[child].sum() / (D.loc[child].sum() + D.loc[parent].sum())))
    for child, parent in SPLIT_ICMR.items():
        f = nd12[child] / (nd12[child] + nd12[parent])
        D.loc[child] = D.loc[parent] * f
        D.loc[parent] = D.loc[parent] * (1 - f)
        adj_log.append((yr, f"{child} split from {parent} (share of total)", f))
    missing = sorted(set(STATES) - set(D.index))
    assert not missing, f"{yr}: missing {missing}"
    return D.loc[STATES]

H = {yr: harmonise(yr) for yr in ANCHOR_YEARS}

# 2) Census date (1 March) -> mid-year (1 July): +4 months at each band's intercensal growth
if DATE_SHIFT:
    g91 = np.log(H[2001] / H[1991]).replace([np.inf, -np.inf], 0).fillna(0) / 10
    g01 = np.log(H[2011] / H[2001]).replace([np.inf, -np.inf], 0).fillna(0) / 10
    G = {1991: g91, 2001: (g91 + g01) / 2, 2011: g01}
    anchor_state = {yr: H[yr] * np.exp(G[yr] * 4 / 12) for yr in ANCHOR_YEARS}
else:
    anchor_state = H
anchor_india = pd.DataFrame({yr: anchor_state[yr].sum() for yr in ANCHOR_YEARS}).T      # year x band

# 3) report adjustment sizes
ns = {yr: cen_long[(cen_long.Year == yr) & cen_long.state.str.startswith("India") & (cen_long.band == "Age not stated")]
                  .TotalFemales.sum() / 1e6 for yr in ANCHOR_YEARS}
print("Age not stated (India, females, M):", {k: round(v, 2) for k, v in ns.items()})
for yr, what, v in adj_log:
    print(f"  {yr}: {what:<48s} {v:.4f}")
for yr in ANCHOR_YEARS:
    print(f"  {yr}: India total raw Census = {census.loc[yr].sum()/1e6:7.2f}M | harmonised (+J&K 1991) = "
          f"{H[yr].sum().sum()/1e6:7.2f}M | after date shift = {anchor_india.loc[yr].sum()/1e6:7.2f}M")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
y = np.arange(len(BANDS))
for ax, yr in zip(axs, ANCHOR_YEARS):
    ax.barh(y - 0.2, H[yr].sum() / census.loc[yr], height=0.4, color=sp.L2, label="harmonised / raw Census")
    ax.barh(y + 0.2, anchor_india.loc[yr] / census.loc[yr], height=0.4, color=sp.OUTC, label="+ date shift / raw Census")
    ax.axvline(1, color="black", lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(BANDS); sp.lock_ticks(ax, "y")
    ax.set_xlim(0.95, 1.06); ax.set_xlabel("Ratio to raw Census India [-]"); ax.set_ylabel("Age band")
    ax.set_title(f"{yr}: effect of harmonisation"); ax.grid(alpha=0.25)
axs[0].legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout(); px.save(fig, "exp3_1_anchor_adjustments"); plt.show()

### 4.2 National Track B: the band-wise ratio method

$$P^B_b(t) = W_b(t)\,\rho_b(t), \qquad \rho_b(t_a) = \frac{C_b(t_a)}{W_b(t_a)}\quad t_a\in\{1991, 2001, 2011\}$$

$\log\rho_b$ is interpolated with a monotone cubic (PCHIP) between the anchors. The variants differ outside 1991–2011 and in
whether $\rho$ is smoothed across neighbouring bands:

| Variant | Outside 1991–2011 | Across bands | Passes exactly through the anchors? |
|---|---|---|---|
| **B-hold** | $\rho$ held at the nearest anchor value | raw | yes |
| **B-taper** | $\log\rho$ fades to 0 (i.e. to WPP) over `T_TAPER` years (cosine) | raw | yes |
| **B-smooth** | held | $\log\rho$ smoothed with weights ¼–½–¼ across adjacent bands | no (by design) |

Holding $\rho$ keeps the Census level for ever; tapering lets the series converge to WPP; smoothing limits jumps in the
correction as a cohort moves from one band to the next.

In [ ]:
from scipy.interpolate import PchipInterpolator
T_TAPER = 30
A = np.array(ANCHOR_YEARS)

def cos_fade(k, T):
    return np.where(k >= T, 0.0, 0.5 * (1 + np.cos(np.pi * np.clip(k, 0, T) / T)))

def log_rho_path(logr_anchor, mode):
    # logr_anchor: array (3,) or (3, n) at the anchor years -> (len(YEARS), n)
    f = PchipInterpolator(A, logr_anchor, axis=0)
    out = f(np.clip(YEARS, A.min(), A.max()))
    if mode == "taper":
        w = np.ones(len(YEARS))
        w[YEARS > A.max()] = cos_fade(YEARS[YEARS > A.max()] - A.max(), T_TAPER)
        w[YEARS < A.min()] = cos_fade(A.min() - YEARS[YEARS < A.min()], T_TAPER)
        out = out * (w[:, None] if out.ndim == 2 else w)
    return out

log_rho = np.log(anchor_india / wpp.loc[A])                       # anchor year x band
kern = np.array([0.25, 0.5, 0.25])
def smooth_bands(v):
    p = np.pad(v, 1, mode="edge")
    return np.convolve(p, kern, mode="valid")
log_rho_sm = log_rho.apply(lambda r: pd.Series(smooth_bands(r.values), index=BANDS), axis=1)

RHO = {"B-hold":   pd.DataFrame(np.exp(log_rho_path(log_rho.values, "hold")),     index=YEARS, columns=BANDS),
       "B-taper":  pd.DataFrame(np.exp(log_rho_path(log_rho.values, "taper")),    index=YEARS, columns=BANDS),
       "B-smooth": pd.DataFrame(np.exp(log_rho_path(log_rho_sm.values, "hold")),  index=YEARS, columns=BANDS)}
trackB_nat = {v: wpp.loc[YEARS] * r for v, r in RHO.items()}
BSTYLE = {"B-hold": dict(color=sp.L2, lw=1.8), "B-taper": dict(color=sp.OUTC, lw=1.6, ls="--"),
          "B-smooth": dict(color=sp.ACC, lw=1.4, ls="-.")}

for v, P in trackB_nat.items():
    err = ((P.loc[A] / anchor_india - 1).abs().max().max()) * 100
    print(f"{v:9s} max |error at anchors| = {err:8.4f} %   total: 1950 {P.loc[1950].sum()/1e6:6.1f}M  "
          f"2011 {P.loc[2011].sum()/1e6:6.1f}M  2050 {P.loc[2050].sum()/1e6:6.1f}M  2100 {P.loc[2100].sum()/1e6:6.1f}M")

#### 4.2a The correction factors $\rho_b(t)$

$\rho=1$ means Track B equals WPP. Black dots are the anchor values.

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(15, 11))
for ax, b in zip(axs.flat, BANDS):
    for v, r in RHO.items():
        ax.plot(r.index, r[b], label=v, **BSTYLE[v])
    ax.scatter(A, np.exp(log_rho[b]), color="black", s=22, zorder=6, label="Census/WPP at anchors")
    ax.axhline(1, color=sp.GREY, lw=0.8, ls=":")
    px.style_pop_axis(ax, f"Age band {b}", ylabel="Correction ρ [-]")
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("3.2a  Census/WPP correction factor by band", y=1.0)
plt.tight_layout(); px.save(fig, "exp3_2a_rho"); plt.show()

#### 4.2b India, all 16 bands: Track B variants with WPP, Census anchors and Track A

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(15, 11))
for ax, b in zip(axs.flat, BANDS):
    px.shade_icmr_window(ax, label=(b == BANDS[0]))
    ax.plot(wpp.index, wpp[b], color=sp.GREY, lw=1.0, ls=":", label="WPP India")
    ax.plot(india_var["A1"].index, india_var["A1"][b], color=sp.L1, lw=1.0, alpha=0.6, label="Track A (A1)")
    for v, P in trackB_nat.items():
        ax.plot(P.index, P[b], label=v, **BSTYLE[v])
    ax.scatter(A, anchor_india[b], color="black", s=20, zorder=6, label="Census anchors")
    px.style_pop_axis(ax, f"Age band {b}", ymax=max(wpp[b].max(), trackB_nat["B-hold"][b].max()))
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=7, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("3.2b  India female population by band: Track B", y=1.0)
plt.tight_layout(); px.save(fig, "exp3_2b_india_bands"); plt.show()

#### 4.2c Totals, growth and age shares over time

In [ ]:
fig = plt.figure(figsize=(18, 9))
gs = fig.add_gridspec(2, 3)
axT, axG = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0])
axS = [fig.add_subplot(gs[i, j]) for i, j in [(0, 1), (0, 2), (1, 1)]]
axN = fig.add_subplot(gs[1, 2])
wt = wpp.sum(axis=1)
for ax in (axT, axG):
    px.shade_icmr_window(ax, label=(ax is axT))
axT.plot(wt.index, wt, color=sp.GREY, ls=":", lw=1.2, label="WPP India")
axT.plot(india_var["A1"].index, india_var["A1"].sum(axis=1), color=sp.L1, lw=1.2, label="Track A (A1)")
axG.plot(wt.index, px.growth_pct(wt), color=sp.GREY, ls=":", lw=1.2)
for v, P in trackB_nat.items():
    axT.plot(P.index, P.sum(axis=1), label=v, **BSTYLE[v])
    axG.plot(P.index, px.growth_pct(P.sum(axis=1)), **BSTYLE[v])
axT.scatter(A, anchor_india.sum(axis=1), color="black", s=25, zorder=6, label="Census anchors")
px.style_pop_axis(axT, "India total", ymax=wt.max()); axT.legend(frameon=False, fontsize=8)
px.style_pop_axis(axG, "Annual growth of the total", ylabel="Growth [%/yr]")
for ax, (v, P) in zip(axS, trackB_nat.items()):
    sh = P.div(P.sum(axis=1), axis=0) * 100
    for b, c in zip(BANDS, colors):
        ax.plot(sh.index, sh[b], color=c, lw=1.2, label=b)
    cs_ = anchor_india.div(anchor_india.sum(axis=1), axis=0) * 100
    for b, c in zip(BANDS, colors):
        ax.scatter(A, cs_[b], color=c, s=12, edgecolors="black", linewidths=0.4, zorder=6)
    px.style_pop_axis(ax, f"{v}: age shares", ylabel="Share of females [%]"); ax.set_ylim(0, 19)
axS[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=7, ncol=2)
axN.axis("off")
plt.tight_layout(); px.save(fig, "exp3_2c_totals_shares"); plt.show()

#### 4.2d Growth by band and year

Dashed lines at 1991 and 2011, where interpolation ends and holding/tapering begins. The table compares the growth jump there with WPP's own year-to-year change.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)
norm = TwoSlopeNorm(vmin=-4, vcenter=0, vmax=6)
for ax, (v, P) in zip(axs, trackB_nat.items()):
    Gm = px.growth_pct(P).loc[1951:]
    im = ax.imshow(Gm.T.values, aspect="auto", cmap="RdBu_r", norm=norm,
                   extent=[Gm.index.min() - 0.5, Gm.index.max() + 0.5, len(BANDS) - 0.5, -0.5])
    for t in (1991.5, 2011.5):
        ax.axvline(t, color="black", lw=0.8, ls="--")
    ax.set_yticks(range(len(BANDS))); ax.set_yticklabels(BANDS); sp.lock_ticks(ax, "y")
    ax.set_title(f"{v}: annual growth by band"); ax.set_xlabel("Year"); ax.set_ylabel("Age band")
fig.colorbar(im, ax=axs, shrink=0.85, label="Growth [%/yr]")
px.save(fig, "exp3_2d_growth_heatmaps"); plt.show()

# growth jump at an edge = growth just inside the interpolated range minus growth just outside it
rows = {}
for v, P in {**trackB_nat, "WPP (reference)": wpp}.items():
    g = px.growth_pct(P)
    rows[(v, "1991 edge")] = g.loc[1992] - g.loc[1991]
    rows[(v, "2011 edge")] = g.loc[2012] - g.loc[2011]
edge = pd.DataFrame(rows).T.round(2)
edge["max |jump|"] = edge.abs().max(axis=1)
print("Growth jump (pp/yr) across the anchor edges; WPP's own year-to-year change shown for scale:")
edge

#### 4.2e Cohort tests

The Census itself fails slightly at young ages (child under-count, age misreporting). Track B inherits that at the anchors; the question is whether the method adds problems of its own.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
COH = [(1991, 0, "Aged 00-04 in 1991"), (2011, 0, "Aged 00-04 in 2011"), (1991, 4, "Aged 20-24 in 1991")]
SER = {"WPP India": (wpp, dict(color=sp.GREY, ls=":", marker="s", ms=2.5, lw=1)),
       "Track A (A1)": (india_var["A1"], dict(color=sp.L1, marker="o", ms=3, lw=1.4)),
       **{v: (P, {**BSTYLE[v], "marker": "o", "ms": 3}) for v, P in trackB_nat.items()}}
for ax, (t0, b0, title) in zip(axs, COH):
    for name, (df, st) in SER.items():
        p = cohort_path(df, t0, b0)
        ax.plot(p.index, p.values, label=name, **st)
    ax.axhline(100, color="black", lw=0.8); ax.axhspan(100, 200, color=sp.OUTC, alpha=0.05, lw=0)
    ax.set_ylim(40, 180)
    px.style_pop_axis(ax, title, ylabel="Cohort size [% of starting size]")
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.05), frameon=False)
plt.tight_layout(); px.save(fig, "exp3_2e_cohort_paths"); plt.show()

WINDOWS = [1991, 2001, 2011, 2030, 2060]
mu_tab = {}
for name, (df, _) in SER.items():
    for t in WINDOWS:
        mu_tab[(name, f"{t}->{t+10}")] = -np.log(cohort_survival(df, t)) / 10 * 1000
mu_tab = pd.DataFrame(mu_tab).T.round(1)
mu_tab["min"] = mu_tab.min(axis=1)
print("Implied death rate per 1000 per year by starting band (negative = cohort grows):")
mu_tab

#### 4.2f Age pyramids at selected years

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 9), sharey=True)
y = np.arange(len(BANDS))
for ax, yr in zip(axs.flat, [1950, 1991, 2011, 2036, 2070, 2100]):
    ax.barh(y, trackB_nat["B-hold"].loc[yr] / 1e6, color=sp.L2, alpha=0.25, label="B-hold")
    for v in ["B-taper", "B-smooth"]:
        ax.plot(trackB_nat[v].loc[yr] / 1e6, y, marker="o", ms=3, label=v, **BSTYLE[v])
    ax.plot(wpp.loc[yr] / 1e6, y, color=sp.GREY, lw=1, ls=":", marker="s", ms=2.5, label="WPP India")
    ax.plot(india_var["A1"].loc[yr] / 1e6, y, color=sp.L1, lw=1, marker=".", label="Track A (A1)")
    if yr in anchor_india.index:
        ax.scatter(anchor_india.loc[yr] / 1e6, y, color="black", s=14, zorder=6, label="Census anchor")
    ax.set_yticks(y); ax.set_yticklabels(BANDS); ax.set_title(str(yr))
    ax.set_xlabel("Female pop. [millions]"); ax.set_ylabel("Age band"); ax.grid(alpha=0.25)
h, l = axs[0, 1].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("3.2f  India female age pyramids: Track B", y=1.0)
plt.tight_layout(); px.save(fig, "exp3_2f_pyramids"); plt.show()

### 4.3 How do state age structures evolve?

- **4.3a** Broad-group shares (0–14, 15–49, 60+) for every state at 1991, 2001, 2011, against India.
- **4.3b** *Demographic year*: the WPP-India year whose age split is closest to each state's. If a state behaved like "India a few years
  ahead or behind", this lag would be stable over time.
- **4.3c** Out-of-sample test: predict 2011 from 1991 + 2001 (and 1991 from 2001 + 2011) with four rules and compare with the Census.

Each state is written as a share of the national band, $d_{s,b}(t) = P_{s,b}(t)/P_b(t)$. Rules beyond the last anchor:

| Rule | Beyond the last anchor |
|---|---|
| **hold** | $d_{s,b}$ stays at its last anchor value |
| **drift** | $\log d_{s,b}$ continues its last decade's trend |
| **drift-damped** | as drift, with the trend decaying at a 15-year half-life |
| **lag** | the state's age split = WPP India's split at (year + the state's lag); shares only |

In [ ]:
shareA = {yr: anchor_state[yr].div(anchor_state[yr].sum(axis=1), axis=0) for yr in ANCHOR_YEARS}
shareI = anchor_india.div(anchor_india.sum(axis=1), axis=0)
GROUPS = {"0-14": BANDS[:3], "15-49": BANDS[3:10], "60+": BANDS[12:]}
order_states = shareA[2011][GROUPS["60+"]].sum(axis=1).sort_values().index.tolist()

fig, axs = plt.subplots(1, 3, figsize=(17, 9), sharey=True)
yy = np.arange(len(order_states))
for ax, (g, bands_g) in zip(axs, GROUPS.items()):
    vals = pd.DataFrame({yr: shareA[yr].loc[order_states, bands_g].sum(axis=1) * 100 for yr in ANCHOR_YEARS})
    for i, s in enumerate(order_states):
        ax.plot(vals.loc[s], [i] * 3, color=sp.GREY, lw=0.8, zorder=1)
    for yr, c in zip(ANCHOR_YEARS, [sp.ACC, sp.L2, sp.OUTC]):
        ax.scatter(vals[yr], yy, color=c, s=18, zorder=3, label=str(yr))
        ax.axvline(shareI.loc[yr, bands_g].sum() * 100, color=c, lw=0.8, ls="--")
    ax.set_yticks(yy); ax.set_yticklabels(order_states, fontsize=8); sp.lock_ticks(ax, "y")
    ax.set_xlabel(f"Share aged {g} [%]"); ax.set_title(f"Share aged {g} (dashed = India)"); ax.grid(alpha=0.25)
axs[0].legend(frameon=False, loc="lower right")
plt.tight_layout(); px.save(fig, "exp3_3a_state_broad_groups"); plt.show()

In [ ]:
wpp_sh = wpp.div(wpp.sum(axis=1), axis=0)
def demographic_year(share_vec):
    d = (wpp_sh.sub(share_vec, axis=1)).abs().sum(axis=1)
    return int(d.idxmin()), d.min() * 50        # best WPP year, misallocation % at that year

lag = pd.DataFrame({yr: {s: demographic_year(shareA[yr].loc[s])[0] - yr for s in STATES} for yr in ANCHOR_YEARS})
lag["change 1991->2011"] = lag[2011] - lag[1991]
lagI = {yr: demographic_year(shareI.loc[yr])[0] - yr for yr in ANCHOR_YEARS}
print("India's own lag vs WPP (years):", lagI)

order_lag = lag[2011].sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(15, 5))
xi = np.arange(len(order_lag))
for yr, c, dx in zip(ANCHOR_YEARS, [sp.ACC, sp.L2, sp.OUTC], [-0.25, 0, 0.25]):
    ax.bar(xi + dx, lag.loc[order_lag, yr], width=0.25, color=c, label=f"Census {yr}")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(xi); ax.set_xticklabels(order_lag, rotation=70, ha="right", fontsize=8); sp.lock_ticks(ax, "x")
ax.set_ylabel("Lag vs WPP India [years]\n(+ = older than India)")
ax.set_title("3.3b  Demographic year: which WPP-India year does each state's age split look like?")
ax.legend(frameon=False); ax.grid(alpha=0.25, axis="y")
plt.tight_layout(); px.save(fig, "exp3_3b_demographic_lag"); plt.show()
lag.loc[order_lag].T

In [ ]:
# 3.3c out-of-sample tests
dA = {yr: (anchor_state[yr] / anchor_state[yr].sum()).clip(lower=1e-9) for yr in ANCHOR_YEARS}   # d_{s,b} at anchors (guard zeros)
HALF = 15.0
def extrap_logd(d_near, d_far, years_ahead, mode):
    slope = (np.log(d_near) - np.log(d_far)) / 10
    if mode == "hold":
        return d_near
    if mode == "drift":
        return d_near * np.exp(slope * years_ahead)
    if mode == "drift-damped":
        eff = HALF / np.log(2) * (1 - np.exp(-np.log(2) * years_ahead / HALF))
        return d_near * np.exp(slope * eff)

def misalloc(pred_sh, true_sh):
    return (pred_sh - true_sh).abs().sum(axis=1) * 50                               # % of women in the wrong band

def run_test(target, near, far, true_nat):
    res = {}
    for mode in ["hold", "drift", "drift-damped"]:
        d = extrap_logd(dA[near], dA[far], 10, mode)
        d = d.replace([np.inf, -np.inf], np.nan).fillna(dA[near])
        d = d / d.sum()                                                              # rake: states sum to India
        P = d * true_nat
        res[(mode, "age split")] = misalloc(P.div(P.sum(axis=1), axis=0), shareA[target])
        res[(mode, "total")] = (P.sum(axis=1) / anchor_state[target].sum(axis=1) - 1).abs() * 100
    L = {s: demographic_year(shareA[near].loc[s])[0] - near for s in STATES}
    lag_pred = pd.DataFrame({s: wpp_sh.loc[int(np.clip(target + L[s], 1950, 2100))] for s in STATES}).T
    res[("lag", "age split")] = misalloc(lag_pred, shareA[target])
    return pd.DataFrame(res)

forward = run_test(2011, 2001, 1991, anchor_india.loc[2011])
backward = run_test(1991, 2001, 2011, anchor_india.loc[1991])
summ = pd.DataFrame({"forward 2011: median": forward.median(), "forward 2011: mean": forward.mean(),
                     "backward 1991: median": backward.median(), "backward 1991: mean": backward.mean()}).round(2)
print("Prediction error by rule (age split: % of women in the wrong band; total: |% error|):")
summ

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(17, 5.5), sharey=True)
for ax, (name, T) in zip(axs, [("Predict 2011 from 1991 + 2001", forward), ("Predict 1991 from 2001 + 2011", backward)]):
    o = T[("hold", "age split")].sort_values().index
    xi = np.arange(len(o))
    for (mode, c, dx) in [("hold", sp.L2, -0.3), ("drift", sp.OUTC, -0.1), ("drift-damped", sp.ACC, 0.1), ("lag", sp.L1, 0.3)]:
        ax.bar(xi + dx, T.loc[o, (mode, "age split")], width=0.2, color=c, label=mode)
    ax.set_xticks(xi); ax.set_xticklabels(o, rotation=70, ha="right", fontsize=8); sp.lock_ticks(ax, "x")
    ax.set_ylabel("Age-split error [% of women misplaced]"); ax.set_title(name); ax.grid(alpha=0.25, axis="y")
axs[0].legend(frameon=False)
plt.tight_layout(); px.save(fig, "exp3_3c_out_of_sample"); plt.show()

### 4.4 State series: India's shape with each state's own anchors

$$P^B_{s,b}(t) = P^B_b(t)\cdot \tilde d_{s,b}(t), \qquad \tilde d_{s,b}(t) = \frac{d_{s,b}(t)}{\sum_{s'} d_{s',b}(t)}$$

- $d_{s,b}$ is exact at the three anchors, interpolated (PCHIP on the log) between them, and extended with **hold** or **drift-damped**.
  This is the same as $W_b\,\rho_{s,b}$ with $\rho_{s,b}=\rho_b\,d_{s,b}$: WPP India gives the shape, the state's own Census anchors give
  its size, age profile and drift.
- **Raking:** with India's band totals as the only margin, iterative proportional fitting reduces to dividing each $d$ by its column
  sum, so the states always add up to India. The raking factor shows how far a rule drifts from adding up by itself.
- Migration is not modelled separately; its effect up to 2011 is carried by the Census anchors.

`NAT_FOR_STATES` chooses the national variant.

In [ ]:
NAT_FOR_STATES = "B-hold"
STATE_RULES = ["hold", "drift-damped"]
S, NB_ = len(STATES), len(BANDS)
logd = np.stack([np.log(dA[yr].loc[STATES, BANDS].values) for yr in ANCHOR_YEARS])      # (3, S, B)
f_between = PchipInterpolator(A, logd.reshape(3, -1), axis=0)

def logd_path(rule):
    out = f_between(np.clip(YEARS, A.min(), A.max())).reshape(len(YEARS), S, NB_)
    if rule == "drift-damped":
        s_fwd = (logd[2] - logd[1]) / 10; s_bwd = (logd[0] - logd[1]) / 10          # per year, away from the data
        for i, t in enumerate(YEARS):
            if t > A.max():
                k = t - A.max(); out[i] = logd[2] + s_fwd * HALF / np.log(2) * (1 - np.exp(-np.log(2) * k / HALF))
            elif t < A.min():
                k = A.min() - t; out[i] = logd[0] + s_bwd * HALF / np.log(2) * (1 - np.exp(-np.log(2) * k / HALF))
    return out

trackB_state, rake_factor = {}, {}
for rule in STATE_RULES:
    d = np.exp(logd_path(rule))                                   # (T, S, B)
    colsum = d.sum(axis=1, keepdims=True)                         # (T, 1, B)
    rake_factor[rule] = 1 / colsum[:, 0, :]                       # (T, B)
    d = d / colsum
    nat = trackB_nat[NAT_FOR_STATES].loc[YEARS].values            # (T, B)
    P = d * nat[:, None, :]
    trackB_state[rule] = {s: pd.DataFrame(P[:, i, :], index=YEARS, columns=BANDS) for i, s in enumerate(STATES)}

for rule in STATE_RULES:
    rf = pd.DataFrame(rake_factor[rule], index=YEARS, columns=BANDS)
    tot = sum(trackB_state[rule].values())
    chk = (tot / trackB_nat[NAT_FOR_STATES] - 1).abs().max().max()
    print(f"{rule:13s} raking factor range {rf.min().min():.3f}–{rf.max().max():.3f} | "
          f"sum of states vs India max |diff| = {chk:.1e}")

#### 4.4a Raking factors (band × year)

1.0 = no correction needed.

In [ ]:
fig, axs = plt.subplots(1, len(STATE_RULES), figsize=(16, 4.8), sharey=True)
for ax, rule in zip(np.atleast_1d(axs), STATE_RULES):
    rf = pd.DataFrame(rake_factor[rule], index=YEARS, columns=BANDS)
    im = ax.imshow(rf.T.values, aspect="auto", cmap="RdBu_r", norm=TwoSlopeNorm(vmin=0.9, vcenter=1, vmax=1.1),
                   extent=[YEARS.min() - 0.5, YEARS.max() + 0.5, NB_ - 0.5, -0.5])
    for t in A:
        ax.axvline(t, color="black", lw=0.6, ls="--")
    ax.set_yticks(range(NB_)); ax.set_yticklabels(BANDS); sp.lock_ticks(ax, "y")
    ax.set_title(f"{rule}: raking factor"); ax.set_xlabel("Year"); ax.set_ylabel("Age band")
fig.colorbar(im, ax=axs, shrink=0.85, label="Raking factor [-]")
px.save(fig, "exp3_4a_raking"); plt.show()

#### 4.4b State totals, 1950–2100: Track B (hold, drift-damped) vs Track A, with the harmonised Census anchors

In [ ]:
ncol = 6; nrow = int(np.ceil(S / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(18, 2.6 * nrow))
RSTYLE = {"hold": dict(color=sp.L2, lw=1.6), "drift-damped": dict(color=sp.OUTC, lw=1.4, ls="--")}
for ax, s in zip(axs.flat, STATES):
    px.shade_icmr_window(ax, label=False)
    a1 = state_var["A1"][s].sum(axis=1)
    ax.plot(a1.index, a1, color=sp.L1, lw=1.0, alpha=0.7, label="Track A (A1)")
    for rule in STATE_RULES:
        t_ = trackB_state[rule][s].sum(axis=1)
        ax.plot(t_.index, t_, label=f"B ({rule})", **RSTYLE[rule])
    ax.scatter(A, [anchor_state[yr].loc[s].sum() for yr in A], color="black", s=12, zorder=6, label="Census anchor")
    top = max(a1.max(), max(trackB_state[r][s].sum(axis=1).max() for r in STATE_RULES))
    px.style_pop_axis(ax, s, ylabel="Females", ymax=top); ax.title.set_fontsize(9)
for ax in list(axs.flat)[S:]:
    ax.axis("off")
h, l = axs.flat[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01), frameon=False)
fig.suptitle(f"3.4b  State totals: Track B ({NAT_FOR_STATES} national) vs Track A", y=1.0)
plt.tight_layout(); px.save(fig, "exp3_4b_state_totals"); plt.show()

#### 4.4c How each state ages: shares aged 0–14 and 60+ (state × year)

In [ ]:
rule_show = "hold"
sh60 = pd.DataFrame({s: trackB_state[rule_show][s][GROUPS["60+"]].sum(axis=1) / trackB_state[rule_show][s].sum(axis=1) * 100
                     for s in STATES})
sh014 = pd.DataFrame({s: trackB_state[rule_show][s][GROUPS["0-14"]].sum(axis=1) / trackB_state[rule_show][s].sum(axis=1) * 100
                      for s in STATES})
order_h = sh60.loc[2011].sort_values().index.tolist()
fig, axs = plt.subplots(1, 2, figsize=(17, 9), sharey=True)
for ax, (M, t, cm) in zip(axs, [(sh014, "Share aged 0-14 [%]", "viridis"), (sh60, "Share aged 60+ [%]", "magma_r")]):
    im = ax.imshow(M[order_h].T.values, aspect="auto", cmap=cm,
                   extent=[YEARS.min() - 0.5, YEARS.max() + 0.5, S - 0.5, -0.5])
    for tt in A:
        ax.axvline(tt, color="white", lw=0.6, ls="--")
    ax.set_yticks(range(S)); ax.set_yticklabels(order_h, fontsize=8); sp.lock_ticks(ax, "y")
    ax.set_xlabel("Year"); ax.set_title(f"B ({rule_show}): {t}")
    fig.colorbar(im, ax=ax, shrink=0.8, label=t)
px.save(fig, "exp3_4c_state_age_heatmaps"); plt.show()

#### 4.4d One state in detail

Change `STATE_TO_PLOT` to look at another state.

In [ ]:
STATE_TO_PLOT = "Kerala"
fig, axs = plt.subplots(4, 4, figsize=(15, 11))
ncd = icmr.loc[STATE_TO_PLOT]
for ax, b in zip(axs.flat, BANDS):
    px.shade_icmr_window(ax, label=(b == BANDS[0]))
    ax.plot(ncd.index, ncd[b], color=sp.ACC, lw=2.4, label="ICMR-NCDIR")
    ax.plot(state_var["A1"][STATE_TO_PLOT].index, state_var["A1"][STATE_TO_PLOT][b], color=sp.L1, lw=1, alpha=0.7, label="Track A (A1)")
    for rule in STATE_RULES:
        ax.plot(YEARS, trackB_state[rule][STATE_TO_PLOT][b], label=f"B ({rule})", **RSTYLE[rule])
    ax.scatter(A, [anchor_state[yr].loc[STATE_TO_PLOT, b] for yr in A], color="black", s=18, zorder=6, label="Census anchor")
    top = max(trackB_state["hold"][STATE_TO_PLOT][b].max(), state_var["A1"][STATE_TO_PLOT][b].max())
    px.style_pop_axis(ax, f"Age band {b}", ylabel="Females", ymax=top)
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle(f"3.4d  {STATE_TO_PLOT}: female population by band", y=1.0)
plt.tight_layout(); px.save(fig, f"exp3_4d_state_{STATE_TO_PLOT.replace(' ', '_')}"); plt.show()

#### 4.4e Cohort test for every state: implied death rate 2012→2022

Blue = negative (cohort grows). Some is expected at state level even in real data, because of migration.

In [ ]:
def state_mu(P, t=2012):
    return (-np.log(cohort_survival(P, t)) / 10 * 1000).replace([np.inf, -np.inf], np.nan)
mu_ncdir = pd.DataFrame({s: state_mu(icmr.loc[s]) for s in STATES}).T
mu_B = pd.DataFrame({s: state_mu(trackB_state["hold"][s]) for s in STATES}).T
order_m = mu_ncdir.iloc[:, :4].mean(axis=1).sort_values().index
fig, axs = plt.subplots(1, 2, figsize=(16, 9), sharey=True)
for ax, (M, t) in zip(axs, [(mu_ncdir, "ICMR-NCDIR 2012→2022"), (mu_B, "Track B (hold) 2012→2022")]):
    im = ax.imshow(M.loc[order_m].values, aspect="auto", cmap="RdBu", norm=TwoSlopeNorm(vmin=-40, vcenter=0, vmax=40))
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(M.columns, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(order_m))); ax.set_yticklabels(order_m, fontsize=8)
    sp.lock_ticks(ax, "both"); ax.set_xlabel("Starting age band"); ax.set_title(t)
fig.colorbar(im, ax=axs, shrink=0.8, label="Implied death rate [per 1000 per yr]")
px.save(fig, "exp3_4e_state_cohort_mu"); plt.show()
print(f"Share of (state, band) cells with negative implied mortality: ICMR-NCDIR {(mu_ncdir < 0).mean().mean()*100:.0f}% | "
      f"Track B {(mu_B < 0).mean().mean()*100:.0f}%")

### 4.5 Validation against SRS 2022

SRS 2022 gives female age distributions for India and 22 bigger states. No track uses it, so it is an independent check.
Error = % of women in the wrong band. SRS has its own sampling error of a few tenths of a percentage point.

In [ ]:
srs = px.load_srs2022_female_shares()
srs_states = [s for s in srs.index if s in STATES]
def split_err(P_share, s):
    return (P_share - srs.loc[s]).abs().sum() * 50

val = {}
for s in srs_states:
    r = {}
    for rule in STATE_RULES:
        P = trackB_state[rule][s].loc[2022]; r[f"B states ({rule})"] = split_err(P / P.sum(), s)
    P = state_var["A1"][s].loc[2022]; r["Track A / ICMR-NCDIR"] = split_err(P / P.sum(), s)
    r["Census 2011 unchanged"] = split_err(shareA[2011].loc[s], s)
    val[s] = r
val = pd.DataFrame(val).T
nat_val = {v: (P.loc[2022] / P.loc[2022].sum() - srs.loc["India"]).abs().sum() * 50 for v, P in trackB_nat.items()}
nat_val["Track A / ICMR-NCDIR"] = (india_var["A1"].loc[2022] / india_var["A1"].loc[2022].sum() - srs.loc["India"]).abs().sum() * 50
nat_val["WPP India"] = (wpp.loc[2022] / wpp.loc[2022].sum() - srs.loc["India"]).abs().sum() * 50
print("India 2022 vs SRS (% of women misplaced):", {k: round(v, 2) for k, v in nat_val.items()})
print("\nStates: median / mean error")
print(val.agg(["median", "mean"]).round(2).to_string())

o = val["B states (hold)"].sort_values().index
fig, ax = plt.subplots(figsize=(15, 5))
xi = np.arange(len(o))
for col, c, dx in [("B states (hold)", sp.L2, -0.3), ("B states (drift-damped)", sp.OUTC, -0.1),
                   ("Census 2011 unchanged", sp.GREY, 0.1), ("Track A / ICMR-NCDIR", sp.L1, 0.3)]:
    ax.bar(xi + dx, val.loc[o, col], width=0.2, color=c, label=col)
ax.set_xticks(xi); ax.set_xticklabels(o, rotation=60, ha="right", fontsize=8); sp.lock_ticks(ax, "x")
ax.set_ylabel("Age-split error vs SRS 2022\n[% of women misplaced]"); ax.set_title("3.5  Which series best matches SRS 2022?")
ax.legend(frameon=False, ncol=4); ax.grid(alpha=0.25, axis="y")
plt.tight_layout(); px.save(fig, "exp3_5_srs2022_validation"); plt.show()

In [ ]:
# Visual check for a few states: 2022 age split, SRS vs Track B vs ICMR-NCDIR
SHOW = ["Bihar", "Uttar Pradesh", "Kerala", "Tamil Nadu", "Delhi", "West Bengal"]
fig, axs = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
y = np.arange(NB_)
for ax, s in zip(axs.flat, SHOW):
    ax.plot(srs.loc[s] * 100, y, color="black", marker="o", ms=4, lw=1.4, label="SRS 2022")
    P = trackB_state["hold"][s].loc[2022]; ax.plot(P / P.sum() * 100, y, marker="o", ms=3, label="B (hold)", **RSTYLE["hold"])
    P = trackB_state["drift-damped"][s].loc[2022]; ax.plot(P / P.sum() * 100, y, marker="o", ms=3, label="B (drift-damped)", **RSTYLE["drift-damped"])
    P = state_var["A1"][s].loc[2022]; ax.plot(P / P.sum() * 100, y, color=sp.L1, marker=".", lw=1, label="ICMR-NCDIR / Track A")
    ax.set_yticks(y); ax.set_yticklabels(BANDS); ax.set_title(s)
    ax.set_xlabel("Share of females [%]"); ax.set_ylabel("Age band"); ax.grid(alpha=0.25)
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("3.5b  2022 female age split: SRS vs Track B vs ICMR-NCDIR", y=1.0)
plt.tight_layout(); px.save(fig, "exp3_5b_srs_profiles"); plt.show()

### 4.6 Track B totals vs ICMR-NCDIR totals

Percent difference of each state's Track B total (hold) from its ICMR-NCDIR total, 2012–2036.

In [ ]:
diffT = pd.DataFrame({s: (trackB_state["hold"][s].sum(axis=1).loc[2012:2036] / icmr.loc[s].sum(axis=1) - 1) * 100
                      for s in STATES})
order_d = diffT.loc[2036].sort_values().index
fig, axs = plt.subplots(1, 2, figsize=(17, 9), gridspec_kw={"width_ratios": [2, 1]})
im = axs[0].imshow(diffT[order_d].T.values, aspect="auto", cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-30, vcenter=0, vmax=30),
                   extent=[2011.5, 2036.5, len(order_d) - 0.5, -0.5])
axs[0].set_yticks(range(len(order_d))); axs[0].set_yticklabels(order_d, fontsize=8); sp.lock_ticks(axs[0], "y")
axs[0].set_xlabel("Year"); axs[0].set_title("Track B (hold) total vs ICMR-NCDIR total [%]")
fig.colorbar(im, ax=axs[0], shrink=0.8, label="Difference [%]")
nat_d = (trackB_nat[NAT_FOR_STATES].sum(axis=1).loc[2012:2036] / icmr_india.sum(axis=1) - 1) * 100
axs[1].plot(nat_d.index, nat_d, color=sp.L2, lw=2, label="India")
axs[1].axhline(0, color="black", lw=0.8)
px.style_pop_axis(axs[1], "India: Track B vs ICMR-NCDIR total", ylabel="Difference [%]"); axs[1].legend(frameon=False)
plt.tight_layout(); px.save(fig, "exp3_6_B_vs_ncdir_totals"); plt.show()
diffT.loc[[2012, 2020, 2030, 2036]].T.round(1).sort_values(2036)

### 4.7 Save Track B outputs

Written to `outputs/trackB/`: harmonised Census anchors, national variants, and state series (long format) for each rule.

In [ ]:
OUTB = os.path.join(px.OUT_DIR, "trackB")
os.makedirs(OUTB, exist_ok=True)
pd.concat({yr: anchor_state[yr] for yr in ANCHOR_YEARS}, names=["year", "state"]).to_csv(os.path.join(OUTB, "census_anchors_harmonised.csv"))
with pd.ExcelWriter(os.path.join(OUTB, "trackB_india.xlsx")) as xw:
    for v, P in trackB_nat.items():
        out = P.copy(); out.index.name = "Year"; out["Total"] = out.sum(axis=1); out.to_excel(xw, sheet_name=v)
for rule in STATE_RULES:
    long = pd.concat(trackB_state[rule], names=["state", "year"]).reset_index().melt(
        id_vars=["state", "year"], var_name="band", value_name="females")
    long.to_parquet(os.path.join(OUTB, f"trackB_states_{NAT_FOR_STATES}_{rule}.parquet"), index=False)
print("saved to", OUTB); print(sorted(os.listdir(OUTB)))

### 4.8 Results

- **Anchors:** all 37 units harmonised for 1991, 2001, 2011. Age-not-stated 1.2–2.1 M per year; J&K 1991 imputed at 3.9 M; the date shift
  adds about 0.6 % to India's total.
- **National variants:** B-hold and B-taper pass exactly through the anchors; B-smooth misses them by up to about 12 % in single bands.
  All variants show growth jumps of up to 2.5 pp/yr at 1991 and 1.7 pp/yr at 2011 (WPP's own year-to-year change is ≤ 0.5), because
  the interpolated correction stops changing abruptly there.
- **Census errors carried forward:** Census under-counts young children and shows age heaping, so $\rho$ is low for 00–04 and 50–54.
  Holding it for ever keeps negative implied death rates in those bands (about −15 and −20 per 1000). Tapering removes this in the long
  run (positive everywhere after about 2041; identical to WPP by 2060).
- **State rules:** hold predicts the age split best out of sample (median 3.1 % misplaced); drift-damped predicts state totals best
  (2.8 % vs 4.6 %). The lag model is worst and its lags are unstable.
- **SRS 2022:** Track B misplaces 5.1–5.7 % of women for India and a median 5.7 % for states, against 7.0 % and 7.8 % for the fixed
  ICMR-NCDIR split.
- **Totals vs ICMR-NCDIR:** India's Track B total is 0.4–1.1 % below ICMR-NCDIR over 2012–2036; individual states differ by much more
  by 2036 (from about −69 % to +28 %).

---
## 5. Track C — ICMR-NCDIR totals × Census-anchored age split

$$
P^{C}_{s,b}(t) \;=\; \underbrace{N^{A}_{s}(t)}_{\text{total from Track A}} \;\times\; \underbrace{\frac{P^{B}_{s,b}(t)}{\sum_{b'} P^{B}_{s,b'}(t)}}_{\text{age split from Track B}}
$$

- $N^{A}_s(t)$: the **ICMR-NCDIR state total** for 2012–2036, extended to 1950–2100 as in Track A (`C_TOTALS`, A1-blend by default so
  there are no kinks at 2012 / 2036).
- The age split comes from **Track B** (national variant `B_NAT`, state rule `B_RULE`). India = sum of the 37 states.

### Why this construction

ICMR-NCDIR's state totals are the official projection figures, and many analyses need to agree with them — for example when
numbers are reported per head of population, compared across studies, or combined with other official statistics that use them.
Their fixed age split, however, is not a realistic age structure (Sections 1–3). Track C keeps the part that must agree and
replaces the part that is unrealistic:

| Quantity | Depends on | Under Track C |
|---|---|---|
| State and India totals | the total | **identical to ICMR-NCDIR** in 2012–2036 |
| Any all-ages ratio (e.g. events per 100,000 women) | the total only | **identical** to one computed on ICMR-NCDIR |
| Age-specific populations and ratios | each band | follow the Census-anchored split |
| Age structure over time | the split | changes with time; no 2011→2012 jump |

### What C does not change
- It keeps ICMR-NCDIR's **state totals** as they are, including very fast projected growth in some small units
  (Dadra & Nagar Haveli, Daman & Diu, Manipur, Mizoram).
- It inherits Track B's age-split properties: Census child under-count and age heaping at the anchors, and B's growth jumps at 1991 / 2011.
- Its total follows ICMR-NCDIR while its split follows B, so cohort tests can differ slightly from B's.

### 5.1 Build Track C for every state and check the construction

Change the three choices at the top to build other versions of C.

In [ ]:
B_NAT    = "B-taper"    # national Track B variant supplying the age shape: "B-hold", "B-taper", "B-smooth"
B_RULE   = "hold"       # state rule for Track B shares: "hold" or "drift-damped"
C_TOTALS = "A1-blend"   # Track A variant supplying totals (ICMR-NCDIR inside 2012-2036): "A1" or "A1-blend"

def dtilde(rule):
    d = np.exp(logd_path(rule))                       # (T, S, B): state share of each national band
    return d / d.sum(axis=1, keepdims=True)           # raked so states sum to India
DT = {r: dtilde(r) for r in STATE_RULES}

def build_B_states(nat, rule):
    P = DT[rule] * trackB_nat[nat].loc[YEARS].values[:, None, :]
    return {s: pd.DataFrame(P[:, i, :], index=YEARS, columns=BANDS) for i, s in enumerate(STATES)}

def build_C_states(nat, rule, totals):
    Bst = build_B_states(nat, rule)
    return {s: Bst[s].div(Bst[s].sum(axis=1), axis=0).mul(state_var[totals][s].sum(axis=1), axis=0) for s in STATES}

trackA_st = state_var[C_TOTALS];           trackA_in = india_var[C_TOTALS]
trackB_st = build_B_states(B_NAT, B_RULE); trackB_in = sum(trackB_st.values())
trackC_st = build_C_states(B_NAT, B_RULE, C_TOTALS); trackC_in = sum(trackC_st.values())

# --- construction checks ---
tot_vs_A   = max((trackC_st[s].sum(axis=1) - trackA_st[s].sum(axis=1)).abs().max() for s in STATES)
tot_vs_ndm = max((trackC_st[s].loc[2012:2036].sum(axis=1) / icmr.loc[s].sum(axis=1) - 1).abs().max() for s in STATES) * 100
sh = lambda P: P.div(P.sum(axis=1), axis=0)
split_vs_B = max((sh(trackC_st[s]) - sh(trackB_st[s])).abs().max().max() for s in STATES)
print(f"C total vs A total, all states/years: max |diff| = {tot_vs_A:.2e} persons")
print(f"C total vs ICMR-NCDIR total, 2012-2036:     max |diff| = {tot_vs_ndm:.2e} %")
print(f"C age split vs B age split:            max |diff| = {split_vs_B:.2e} (share)")
print(f"India 2016: A = {trackA_in.loc[2016].sum()/1e6:.2f}M, B = {trackB_in.loc[2016].sum()/1e6:.2f}M, "
      f"C = {trackC_in.loc[2016].sum()/1e6:.2f}M, ICMR-NCDIR = {icmr_india.loc[2016].sum()/1e6:.2f}M")

TRK = {"A: ICMR-NCDIR, frozen split": (trackA_st, trackA_in, dict(color=sp.L1, lw=1.8)),
       "B: Census-anchored":     (trackB_st, trackB_in, dict(color=sp.L2, lw=1.6, ls="--")),
       "C: ICMR-NCDIR totals, B split": (trackC_st, trackC_in, dict(color=sp.OUTC, lw=1.8))}

### 5.2 India: totals, age shares and every band

A and C have identical totals. A's shares are flat; B and C move with the population through the Census anchors and close to SRS 2022.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
px.shade_icmr_window(axs[0])
for name, (_, P, st) in TRK.items():
    axs[0].plot(P.index, P.sum(axis=1), label=name, **st)
axs[0].scatter(icmr_india.index, icmr_india.sum(axis=1), color=sp.ACC, s=10, zorder=6, label="ICMR-NCDIR")
axs[0].scatter(A, anchor_india.sum(axis=1), color="black", s=25, zorder=7, label="Census anchors")
px.style_pop_axis(axs[0], "India total", ymax=trackB_in.sum(axis=1).max()); axs[0].legend(frameon=False, fontsize=8)
for ax, (g, bands_g) in zip(axs[1:], [("0-14", GROUPS["0-14"]), ("60+", GROUPS["60+"])]):
    px.shade_icmr_window(ax, label=False)
    for name, (_, P, st) in TRK.items():
        ax.plot(P.index, P[bands_g].sum(axis=1) / P.sum(axis=1) * 100, label=name, **st)
    ax.scatter(A, anchor_india[bands_g].sum(axis=1) / anchor_india.sum(axis=1) * 100, color="black", s=25, zorder=7, label="Census")
    ax.scatter([2022], [srs.loc["India", bands_g].sum() * 100], color="black", marker="*", s=90, zorder=7, label="SRS 2022")
    px.style_pop_axis(ax, f"India: share of females aged {g}", ylabel="Share [%]")
axs[2].legend(frameon=False, fontsize=8)
plt.tight_layout(); px.save(fig, "exp4_2a_india_totals_shares"); plt.show()

fig, axs = plt.subplots(4, 4, figsize=(15, 11))
for ax, b in zip(axs.flat, BANDS):
    px.shade_icmr_window(ax, label=(b == BANDS[0]))
    for name, (_, P, st) in TRK.items():
        ax.plot(P.index, P[b], label=name, **st)
    ax.scatter(icmr_india.index, icmr_india[b], color=sp.ACC, s=6, zorder=6, label="ICMR-NCDIR")
    ax.scatter(A, anchor_india[b], color="black", s=18, zorder=7, label="Census anchors")
    px.style_pop_axis(ax, f"Age band {b}", ymax=max(P[b].max() for _, P, _ in TRK.values()))
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("4.2b  India female population by band: Tracks A, B, C", y=1.0)
plt.tight_layout(); px.save(fig, "exp4_2b_india_bands"); plt.show()

### 5.3 The 2011→2012 transition

"Census→ICMR-NCDIR stitch" places the Census (to 2011) next to ICMR-NCDIR (from 2012). Track A avoids the jump only by departing from the Census before 2012; Track C keeps ICMR-NCDIR's totals and stays continuous with the Census.

In [ ]:
stitch = pd.concat([anchor_india.reindex(range(1991, 2012)).interpolate(), icmr_india.loc[2012:2036]])
fig, axs = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, b in zip(axs, ["00-04", "10-14", "50-54", "75+"]):
    px.shade_icmr_window(ax, label=(b == "00-04"))
    d = stitch.loc[2000:2025, b]
    ax.plot(d.index, d, color=sp.ACC, lw=1.4, marker=".", label="Census→ICMR-NCDIR stitch")
    for name, (_, P, st) in TRK.items():
        ax.plot(P.loc[2000:2025].index, P.loc[2000:2025, b], label=name, **st)
    ax.scatter(A, anchor_india[b], color="black", s=25, zorder=7, label="Census anchors")
    ax.set_xlim(2000, 2025)
    px.style_pop_axis(ax, f"Age band {b}", ymax=max(stitch[b].max(), trackB_in.loc[2000:2025, b].max()))
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.05), frameon=False)
plt.tight_layout(); px.save(fig, "exp4_3_jump_2012"); plt.show()

jump = pd.DataFrame({name: (P.loc[2012] / P.loc[2011] - 1) * 100 for name, (_, P, _) in TRK.items()})
jump["Census→ICMR-NCDIR stitch"] = (stitch.loc[2012] / stitch.loc[2011] - 1) * 100
print("One-year change 2011→2012 by band [%] (normal year-to-year change is about −2 to +5 %):")
jump.round(1).T

### 5.4 Cohort tests

Implied annual death rate per 1000 by starting band for 10-year windows (negative = cohort grows), and the share of (state, band) cells with negative implied mortality, 2012→2022.

In [ ]:
WIN = [1991, 2001, 2012, 2030]
fig, axs = plt.subplots(1, len(WIN), figsize=(18, 4.6), sharey=True)
x = np.arange(len(BANDS) - 3)
for ax, t in zip(axs, WIN):
    ax.plot(x, -np.log(cohort_survival(wpp, t)) / 10 * 1000, color=sp.GREY, ls=":", marker="s", ms=3, label="WPP India")
    for name, (_, P, st) in TRK.items():
        ax.plot(x, -np.log(cohort_survival(P, t)) / 10 * 1000, marker="o", ms=3, label=name, **st)
    ax.axhline(0, color="black", lw=0.8); ax.axhspan(-60, 0, color=sp.OUTC, alpha=0.06, lw=0); ax.set_ylim(-35, 45)
    ax.set_xticks(x); ax.set_xticklabels(BANDS[:len(x)], rotation=60, ha="right", fontsize=8); sp.lock_ticks(ax, "x")
    ax.set_xlabel("Starting age band"); ax.set_ylabel("Implied death rate [per 1000 per yr]"); ax.set_title(f"{t} → {t+10}")
    ax.grid(alpha=0.25)
axs[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); px.save(fig, "exp4_4_cohort_mu"); plt.show()

def neg_share(stdict, t=2012):
    M = pd.DataFrame({s: (-np.log(cohort_survival(stdict[s], t)) / 10 * 1000).replace([np.inf, -np.inf], np.nan)
                      for s in STATES}).T
    return (M < 0).mean().mean() * 100
print("States, 2012→2022: share of (state, band) cells with negative implied mortality")
for name, (stdict, _, _) in TRK.items():
    print(f"  {name:28s} {neg_share(stdict):5.1f} %")

### 5.5 States

Share of women aged 60+ in every state: A (flat) vs B and C, with Census anchors (dots) and SRS 2022 (stars, bigger states only). Then all 16 bands for one state (change `STATE_TO_PLOT_C`).

In [ ]:
ncol = 6; nrow = int(np.ceil(S / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(18, 2.6 * nrow))
for ax, s in zip(axs.flat, STATES):
    px.shade_icmr_window(ax, label=False)
    for name, (stdict, _, st) in TRK.items():
        P = stdict[s]; ax.plot(P.index, P[GROUPS["60+"]].sum(axis=1) / P.sum(axis=1) * 100, label=name, **st)
    ax.scatter(A, [anchor_state[y].loc[s, GROUPS["60+"]].sum() / anchor_state[y].loc[s].sum() * 100 for y in A],
               color="black", s=12, zorder=7, label="Census")
    if s in srs.index:
        ax.scatter([2022], [srs.loc[s, GROUPS["60+"]].sum() * 100], color="black", marker="*", s=50, zorder=7, label="SRS 2022")
    px.style_pop_axis(ax, s, ylabel="60+ share [%]"); ax.title.set_fontsize(9)
for ax in list(axs.flat)[S:]:
    ax.axis("off")
h, l = axs.flat[1].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.01), frameon=False)
fig.suptitle("4.6a  Share of women aged 60+ by state: A (frozen) vs B vs C", y=1.0)
plt.tight_layout(); px.save(fig, "exp4_6a_states_60plus"); plt.show()

In [ ]:
STATE_TO_PLOT_C = "Bihar"
fig, axs = plt.subplots(4, 4, figsize=(15, 11))
for ax, b in zip(axs.flat, BANDS):
    px.shade_icmr_window(ax, label=(b == BANDS[0]))
    for name, (stdict, _, st) in TRK.items():
        ax.plot(YEARS, stdict[STATE_TO_PLOT_C][b], label=name, **st)
    ax.scatter(icmr.loc[STATE_TO_PLOT_C].index, icmr.loc[STATE_TO_PLOT_C][b], color=sp.ACC, s=6, zorder=6, label="ICMR-NCDIR")
    ax.scatter(A, [anchor_state[y].loc[STATE_TO_PLOT_C, b] for y in A], color="black", s=18, zorder=7, label="Census anchors")
    px.style_pop_axis(ax, f"Age band {b}", ylabel="Females", ymax=max(d[STATE_TO_PLOT_C][b].max() for d, _, _ in TRK.values()))
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle(f"4.6b  {STATE_TO_PLOT_C}: Tracks A, B, C by band", y=1.0)
plt.tight_layout(); px.save(fig, f"exp4_6b_{STATE_TO_PLOT_C.replace(' ', '_')}"); plt.show()

### 5.6 Sensitivity to the Track B choices

Every combination of national variant × state rule gives a version of C with **identical totals**; only the age split changes.

In [ ]:
SENS = {(nat, rule): sum(build_C_states(nat, rule, C_TOTALS).values()) for nat in trackB_nat for rule in STATE_RULES}
fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))
for (nat, rule), P in SENS.items():
    lab = f"{nat}, {rule}"
    kw = dict(color=BSTYLE[nat]["color"], ls="-" if rule == "hold" else "--", lw=1.4, label=lab)
    for ax, g in zip(axs, ["0-14", "15-49", "60+"]):
        ax.plot(P.index, P[GROUPS[g]].sum(axis=1) / P.sum(axis=1) * 100, **kw)
for ax, g in zip(axs, ["0-14", "15-49", "60+"]):
    ax.plot(trackA_in.index, trackA_in[GROUPS[g]].sum(axis=1) / trackA_in.sum(axis=1) * 100, color=sp.L1, lw=2, label="Track A")
    px.shade_icmr_window(ax, label=False)
    px.style_pop_axis(ax, f"C variants: India share aged {g}", ylabel="Share [%]")
axs[2].legend(frameon=False, fontsize=7)
plt.tight_layout(); px.save(fig, "exp4_7_sensitivity"); plt.show()

### 5.7 Save Track C outputs

Written to `outputs/trackC/`: India series (xlsx) and state series (long parquet), tagged with the choices used.

In [ ]:
OUTC_DIR = os.path.join(px.OUT_DIR, "trackC")
os.makedirs(OUTC_DIR, exist_ok=True)
tag = f"{B_NAT}_{B_RULE}_{C_TOTALS}"
out = trackC_in.copy(); out.index.name = "Year"; out["Total"] = out.sum(axis=1)
out.to_excel(os.path.join(OUTC_DIR, f"trackC_india_{tag}.xlsx"))
long = pd.concat(trackC_st, names=["state", "year"]).reset_index().melt(id_vars=["state", "year"], var_name="band", value_name="females")
long.to_parquet(os.path.join(OUTC_DIR, f"trackC_states_{tag}.parquet"), index=False)
print("saved to", OUTC_DIR); print(sorted(os.listdir(OUTC_DIR)))

### 5.8 Results (defaults: B-taper split, hold rule, A1-blend totals)

- **Construction is exact:** C's totals equal Track A / ICMR-NCDIR for every state and year (difference < 1e-7 persons), and its age
  split equals Track B's.
- **2011→2012:** stitching the Census to ICMR-NCDIR changes 00–04 by −18 % and 75+ by +71 % in one year; Track A changes every band by
  the same 1.4 %; Track C changes bands by −0.8 % to +5.4 %, a normal one-year pattern.
- **Age structure:** C's share aged 0–14 and 60+ follows the Census anchors and lies close to SRS 2022; A's stays fixed.
- **Cohort test:** negative implied mortality in about 42 % of state-band cells (A 47 %, B 38 %); state-level migration and Census errors
  contribute to all three.
- **Sensitivity:** the Track B choices behind C change the age split only modestly compared with the difference between A and C.

---
## 6. Comparison of the three tracks

| # | Criterion | Why it matters |
|---|---|---|
| 1 | Totals vs ICMR-NCDIR, 2012–2036 | Agreement with the official projection totals |
| 2 | Age split vs Census (1991, 2001, 2011) | Agreement with the official counts |
| 3 | Total vs Census (India, 1991) | Plausibility of the backward extension |
| 4 | Age split vs SRS 2022 (independent) | Out-of-sample realism |
| 5 | Cohort consistency (negative implied mortality, states 2012→2022) | Suitability for fitting births → aging → deaths models |
| 6 | Largest growth kink at the joins (1991, 2011, 2012, 2036) | Smoothness |
| 7 | Change in the 60+ share 1950→2100 | Whether the age structure evolves at all (information, not a score) |

In [ ]:
def misplaced(P_share, ref_share):
    return (P_share - ref_share).abs().sum() * 50

def max_kink(P, years=(1991, 2011, 2012, 2036)):
    g = px.growth_pct(P)
    return max((g.loc[t + 1] - g.loc[t]).abs().max() for t in years)

def change_60(P):
    return (sh(P).loc[2100, GROUPS["60+"]].sum() - sh(P).loc[1950, GROUPS["60+"]].sum()) * 100

score = {}
for name, (stdict, P, _) in TRK.items():
    r = {}
    r["1 totals vs ICMR-NCDIR 2012-36: max |%| (states)"] = max(
        (stdict[s].loc[2012:2036].sum(axis=1) / icmr.loc[s].sum(axis=1) - 1).abs().max() for s in STATES) * 100
    r["2 age split vs Census: median % misplaced (states, 3 yrs)"] = np.median(
        [misplaced(sh(stdict[s]).loc[y], anchor_state[y].loc[s] / anchor_state[y].loc[s].sum()) for s in STATES for y in A])
    r["3 India total vs Census 1991: %"] = (P.loc[1991].sum() / anchor_india.loc[1991].sum() - 1) * 100
    r["4 age split vs SRS 2022: India % misplaced"] = misplaced(sh(P).loc[2022], srs.loc["India"])
    r["4 age split vs SRS 2022: states median %"] = np.median([misplaced(sh(stdict[s]).loc[2022], srs.loc[s])
                                                               for s in STATES if s in srs.index])
    r["5 negative implied mortality: % of state-band cells"] = neg_share(stdict)
    r["6 largest growth kink at joins: pp/yr (India)"] = max_kink(P)
    r["7 change in 60+ share 1950→2100: pp (India)"] = change_60(P)
    score[name.split(":")[0]] = r
score = pd.DataFrame(score)
score["WPP (reference)"] = np.nan
score.loc["4 age split vs SRS 2022: India % misplaced", "WPP (reference)"] = misplaced(sh(wpp).loc[2022], srs.loc["India"])
score.loc["6 largest growth kink at joins: pp/yr (India)", "WPP (reference)"] = max_kink(wpp)
score.loc["7 change in 60+ share 1950→2100: pp (India)", "WPP (reference)"] = change_60(wpp)
score.round(2)

### 6.1 Scorecard

Colour = rank within each row among A, B, C (green best, red worst; row 7 is information only, grey). Smaller is better for rows 1–6 (row 3 by absolute value).

In [ ]:
tracks = ["A", "B", "C"]
vals = score[tracks]
rankable = [i for i in vals.index if not i.startswith("7")]
rank = vals.loc[rankable].abs().round(6).rank(axis=1, method="min")
fig, ax = plt.subplots(figsize=(12, 6.5))
cmap = plt.get_cmap("RdYlGn_r")
for i, row in enumerate(vals.index):
    for j, t in enumerate(tracks):
        if row in rankable:
            c = cmap((rank.loc[row, t] - 1) / 2 * 0.8 + 0.1)
        else:
            c = (0.9, 0.9, 0.9, 1)
        ax.add_patch(plt.Rectangle((j, i), 1, 1, color=c, ec="white", lw=2))
        ax.text(j + 0.5, i + 0.5, f"{vals.loc[row, t]:.2f}", ha="center", va="center", fontsize=10)
ax.set_xlim(0, 3); ax.set_ylim(len(vals), 0)
ax.set_xticks(np.arange(3) + 0.5); ax.set_xticklabels(["Track A\n(ICMR-NCDIR, frozen)", "Track B\n(Census-anchored)", "Track C\n(ICMR-NCDIR totals, B split)"])
ax.set_yticks(np.arange(len(vals)) + 0.5); ax.set_yticklabels(vals.index, fontsize=9)
sp.lock_ticks(ax, "both"); ax.tick_params(length=0)
for sp_ in ax.spines.values():
    sp_.set_visible(False)
ax.set_title("5.1  Scorecard: A vs B vs C (green = best in row)")
plt.tight_layout(); px.save(fig, "exp5_1_scorecard"); plt.show()

### 6.2 Dashboard (India)

Total, age shares, growth and the cohort test in one figure.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 9))
ax = axs[0, 0]; px.shade_icmr_window(ax)
for name, (_, P, st) in TRK.items():
    ax.plot(P.index, P.sum(axis=1), label=name, **st)
ax.scatter(A, anchor_india.sum(axis=1), color="black", s=22, zorder=7, label="Census")
px.style_pop_axis(ax, "Total female population", ymax=trackB_in.sum(axis=1).max()); ax.legend(frameon=False, fontsize=8)
for ax, g in zip([axs[0, 1], axs[0, 2], axs[1, 0]], ["0-14", "60+", "15-49"]):
    px.shade_icmr_window(ax, label=False)
    for name, (_, P, st) in TRK.items():
        ax.plot(P.index, sh(P)[GROUPS[g]].sum(axis=1) * 100, **st)
    ax.scatter(A, anchor_india[GROUPS[g]].sum(axis=1) / anchor_india.sum(axis=1) * 100, color="black", s=22, zorder=7)
    ax.scatter([2022], [srs.loc["India", GROUPS[g]].sum() * 100], color="black", marker="*", s=80, zorder=7)
    px.style_pop_axis(ax, f"Share aged {g}", ylabel="Share [%]")
ax = axs[1, 1]; px.shade_icmr_window(ax, label=False)
for name, (_, P, st) in TRK.items():
    ax.plot(P.index, px.growth_pct(P.sum(axis=1)), **st)
px.style_pop_axis(ax, "Annual growth of the total", ylabel="Growth [%/yr]")
ax = axs[1, 2]
x = np.arange(len(BANDS) - 3)
ax.plot(x, -np.log(cohort_survival(wpp, 2012)) / 10 * 1000, color=sp.GREY, ls=":", marker="s", ms=3, label="WPP")
for name, (_, P, st) in TRK.items():
    ax.plot(x, -np.log(cohort_survival(P, 2012)) / 10 * 1000, marker="o", ms=3, **st)
ax.axhline(0, color="black", lw=0.8); ax.axhspan(-60, 0, color=sp.OUTC, alpha=0.06, lw=0); ax.set_ylim(-30, 45)
ax.set_xticks(x); ax.set_xticklabels(BANDS[:len(x)], rotation=60, ha="right", fontsize=8); sp.lock_ticks(ax, "x")
ax.set_xlabel("Starting age band"); ax.set_ylabel("per 1000 per yr"); ax.set_title("Implied death rate 2012→2022"); ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); px.save(fig, "exp5_2_dashboard"); plt.show()

### 6.3 Summary

| | Track A | Track B | Track C |
|---|---|---|---|
| Agrees with ICMR-NCDIR totals | **yes** | no (up to about 68 % off for small units) | **yes** |
| Agrees with Census age split at 1991 / 2001 / 2011 | no (about 14 % misplaced) | **yes** | **yes** |
| Matches independent SRS 2022 age split | weakest (7.0 % India, 7.8 % states) | good (5.4 / 5.3 %) | good (5.3 / 5.3 %) |
| Age structure evolves over time | no | **yes** | **yes** |
| Cohort consistency | weakest (47 %) | **best** (38 %) | middle (42 %) |
| Smoothness at the joins | **best** (0.07 pp) | 2.5 pp at 1991 / 2011 | 2.6 pp at 1991 / 2011 |

**Which track for which use**

- **Track C** when results must agree with ICMR-NCDIR's official totals but also need a realistic age structure.
- **Track B** when agreement with the Census and demographic consistency matter more than agreement with ICMR-NCDIR totals —
  e.g. for fitting population-dynamics models or for the pre-2012 history.
- **Track A** only when ICMR-NCDIR's own age split has to be reproduced exactly.

**Known limitations**
- The growth jumps at 1991 / 2011 in Tracks B and C come from the correction factor stopping abruptly at the Census anchors; a
  slope-continuing blend (as used for A1-blend) would remove them.
- State totals after 2036 (Tracks A and C) and all state series (Track B) use India's WPP shape; states have no separate WPP projection.
- Census child under-count and age heaping are carried into Tracks B and C at the anchors.

---
## 7. Export tidy CSV files

One long-format CSV per series, easy to use from any language. Columns: `track`, `variant`, `unit` (India or a state/UT),
`year`, `band`, `females` (persons). Written to `outputs/csv/`.

In [ ]:
CSV = os.path.join(px.OUT_DIR, "csv"); os.makedirs(CSV, exist_ok=True)
def tidy(df, track, variant, unit):
    t = df.copy(); t.index.name = "year"
    t = t.reset_index().melt(id_vars="year", var_name="band", value_name="females")
    t.insert(0, "unit", unit); t.insert(0, "variant", variant); t.insert(0, "track", track)
    return t

india = pd.concat([tidy(P, "A", v, "India") for v, P in india_var.items()] +
                  [tidy(P, "B", v, "India") for v, P in trackB_nat.items()] +
                  [tidy(trackC_in, "C", f"{B_NAT}_{B_RULE}_{C_TOTALS}", "India")])
india.to_csv(os.path.join(CSV, "india_female_population_by_age_1950_2100.csv"), index=False)

states = {"A": ("A1", state_var["A1"]),
          "B": (f"{NAT_FOR_STATES}_hold", trackB_state["hold"]),
          "C": (f"{B_NAT}_{B_RULE}_{C_TOTALS}", trackC_st)}
for trk, (v, d) in states.items():
    pd.concat([tidy(d[s], trk, v, s) for s in STATES]).to_csv(
        os.path.join(CSV, f"states_female_population_by_age_1950_2100_track{trk}.csv"), index=False)
print(sorted(os.listdir(CSV)))